# Run the Wonderland Pipeline From Zero Setup to Result

**What this section does:** introduces the notebook, defines the practical end-to-end workflow that is actually implemented in this repository, and clarifies what counts as a result versus an evaluation.

This notebook is a **repo-specific onboarding and execution guide** for the current `rag-from-scratch` codebase. It is designed for a new engineer to run the checked-in workflow from local setup through profiling, routing, baseline evaluation, synthetic-data generation, SFT-data preparation, and optional LoRA submission packaging.

## What “final result” means in this repo
Because this repository does **not** yet include a checked-in trainer, vLLM inference runner, or hidden-test prediction script, the most honest “final result” you can produce today is:

1. a verified offline benchmark profile,
2. a verified router report,
3. a baseline evaluation matrix with per-baseline reports and a structured comparison object,
4. optional synthetic and SFT artifacts for future LoRA work, and
5. an optional packaged LoRA submission bundle **if you already have a Nemotron-compatible adapter directory**.

## What “evaluation” means in this repo
Evaluation in the current repo means running the **real checked-in offline evaluation path** in `scripts/eval_baselines.py`, which reports:
- overall exact-match accuracy,
- answer-format accuracy,
- per-family accuracy, and
- random-vs-structure-aware split comparisons.

> **Important assumption:** the visible `test.csv` is only a smoke-test sample and overlaps with `train.csv`, so it is **not** a trustworthy validation set for experiment selection.


## 1. Environment and prerequisites

**What this section does:** explains the assumptions needed to run the notebook from top to bottom and highlights which steps are lightweight versus optional.

### Python assumptions
- Recommended: **Python 3.10+**.
- The checked-in project scripts use the **Python standard library only**.
- To run this file as a notebook, you need Jupyter locally or a Kaggle notebook runtime.

### Minimal environment setup
Local Jupyter path:
```bash
python -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip notebook
jupyter notebook notebooks/run_full_pipeline.ipynb
```

Kaggle path:
- Attach the competition dataset so `/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/` is available.
- Open this notebook directly in Kaggle.
- **Do not upload extra `.py` modules.** This notebook will materialize the required script files into `/kaggle/working/rag-from-scratch/` automatically.

### GPU / CUDA assumptions
- **No GPU is required** for the implemented workflow in this notebook.
- GPU, CUDA, PEFT, PyTorch, and vLLM only become relevant when you train or serve a real Nemotron LoRA outside the currently checked-in code.

### Path assumptions
- **Local mode:** run Jupyter from the repository root so local relative paths resolve naturally.
- **Kaggle mode:** the notebook auto-detects Kaggle, copies `train.csv` / `test.csv` into a writable workspace, and writes embedded script sources there so every later cell can run unchanged.

### Required inputs
This notebook needs:
- `train.csv`
- `test.csv`
- the checked-in workflow scripts, either from the local repo or from the notebook's embedded copies


In [ ]:
from __future__ import annotations

import csv
import json
import os
import platform
import shutil
import subprocess
import sys
from itertools import islice
from pathlib import Path
from pprint import pprint

SOURCE_ROOT = Path.cwd().resolve()
KAGGLE_INPUT_ROOT = Path('/kaggle/input')
KAGGLE_WORKING_ROOT = Path('/kaggle/working')
COMPETITION_SLUG = 'nvidia-nemotron-model-reasoning-challenge'
COMPETITION_INPUT_CANDIDATES = [
    KAGGLE_INPUT_ROOT / 'competitions' / COMPETITION_SLUG,
    KAGGLE_INPUT_ROOT / COMPETITION_SLUG,
]
MODEL_INPUT_CANDIDATES = [
    KAGGLE_INPUT_ROOT / 'models' / 'metric' / 'nemotron-3-nano-30b-a3b-bf16' / 'transformers' / 'default' / '1',
]
EMBEDDED_SCRIPT_SOURCES = json.loads(r'''{
  "scripts/profile_dataset.py": "#!/usr/bin/env python3\n\"\"\"Profile the Wonderland reasoning benchmark for reverse-engineering work.\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport re\nimport statistics\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\nfrom typing import Any\n\nROOT = Path(__file__).resolve().parents[1]\nTRAIN_PATH = ROOT / \"train.csv\"\nTEST_PATH = ROOT / \"test.csv\"\n\nFAMILY_RULES = [\n    (\n        \"bit_transform\",\n        \"8-bit bit transform\",\n        \"In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers.\",\n    ),\n    (\n        \"text_cipher\",\n        \"text decryption\",\n        \"In Alice's Wonderland, secret encryption rules are used on text.\",\n    ),\n    (\n        \"roman_numeral\",\n        \"roman numeral conversion\",\n        \"In Alice's Wonderland, numbers are secretly converted into a different numeral system.\",\n    ),\n    (\n        \"unit_conversion\",\n        \"linear unit conversion\",\n        \"In Alice's Wonderland, a secret unit conversion is applied to measurements.\",\n    ),\n    (\n        \"gravity\",\n        \"quadratic gravity\",\n        \"In Alice's Wonderland, the gravitational constant has been secretly changed.\",\n    ),\n    (\n        \"equation_transform\",\n        \"equation transform\",\n        \"In Alice's Wonderland, a secret set of transformation rules is applied to equations.\",\n    ),\n]\n\nNUMERIC_EXPR_RE = re.compile(r\"^[0-9]{2}([^A-Za-z0-9\\s])[0-9]{2}$\")\nBINARY_ANSWER_RE = re.compile(r\"^[01]{8}$\")\nROMAN_RE = re.compile(r\"^[IVXLCDM]+$\")\nDECIMAL_2_RE = re.compile(r\"^-?\\d+\\.\\d{2}$\")\nDECIMAL_OTHER_RE = re.compile(r\"^-?\\d+\\.\\d+$\")\nINTEGER_RE = re.compile(r\"^-?\\d+$\")\nLOWER_TEXT_RE = re.compile(r\"^[a-z ]+$\")\n\n\ndef read_csv(path: Path) -> list[dict[str, str]]:\n    with path.open(newline=\"\", encoding=\"utf-8\") as handle:\n        return list(csv.DictReader(handle))\n\n\ndef quantile(values: list[float], q: float) -> float:\n    if not values:\n        return 0.0\n    if len(values) == 1:\n        return float(values[0])\n    return statistics.quantiles(values, n=100, method=\"inclusive\")[int(q * 100) - 1]\n\n\ndef summarize_lengths(values: list[int]) -> dict[str, float]:\n    return {\n        \"min\": min(values),\n        \"p10\": quantile(values, 0.10),\n        \"median\": statistics.median(values),\n        \"p90\": quantile(values, 0.90),\n        \"max\": max(values),\n    }\n\n\ndef classify_family(prompt: str) -> tuple[str, str]:\n    first_line = prompt.splitlines()[0]\n    for family_id, label, prefix in FAMILY_RULES:\n        if first_line.startswith(prefix):\n            return family_id, label\n    return \"unknown\", first_line\n\n\ndef classify_answer(answer: str) -> str:\n    if BINARY_ANSWER_RE.fullmatch(answer):\n        return \"binary8\"\n    if ROMAN_RE.fullmatch(answer):\n        return \"roman\"\n    if DECIMAL_2_RE.fullmatch(answer):\n        return \"decimal_2dp\"\n    if DECIMAL_OTHER_RE.fullmatch(answer):\n        return \"decimal_other\"\n    if INTEGER_RE.fullmatch(answer):\n        return \"integer\"\n    if LOWER_TEXT_RE.fullmatch(answer):\n        return \"lowercase_text\"\n    return \"symbolic\"\n\n\ndef example_count(prompt: str, family_id: str) -> int:\n    lines = prompt.splitlines()\n    if family_id in {\"bit_transform\", \"text_cipher\", \"roman_numeral\"}:\n        return sum(\" -> \" in line for line in lines)\n    if family_id == \"unit_conversion\":\n        return sum(\" becomes \" in line for line in lines)\n    if family_id == \"gravity\":\n        return sum(line.startswith(\"For t =\") for line in lines)\n    if family_id == \"equation_transform\":\n        return sum(\" = \" in line for line in lines)\n    return 0\n\n\ndef analyze_text_family(rows: list[dict[str, str]]) -> dict[str, Any]:\n    plaintext_vocab = Counter()\n    answer_length_counter = Counter()\n    unseen_answer_word_counter = Counter()\n    for row in rows:\n        seen_plain = set()\n        for line in row[\"prompt\"].splitlines():\n            if \" -> \" in line:\n                _, plain = line.split(\" -> \", 1)\n                words = plain.split()\n                plaintext_vocab.update(words)\n                seen_plain.update(words)\n        answer_words = row[\"answer\"].split()\n        answer_length_counter[len(answer_words)] += 1\n        unseen = sum(word not in seen_plain for word in answer_words)\n        unseen_answer_word_counter[unseen] += 1\n    return {\n        \"plaintext_vocab_size\": len(plaintext_vocab),\n        \"top_plaintext_tokens\": plaintext_vocab.most_common(25),\n        \"answer_word_lengths\": dict(answer_length_counter),\n        \"unseen_answer_words_per_prompt\": dict(unseen_answer_word_counter),\n    }\n\n\ndef analyze_roman_family(rows: list[dict[str, str]]) -> dict[str, Any]:\n    targets = []\n    for row in rows:\n        match = re.search(r\"write the number (\\d+) in the Wonderland numeral system\", row[\"prompt\"])\n        if match:\n            targets.append(int(match.group(1)))\n    return {\n        \"target_min\": min(targets),\n        \"target_max\": max(targets),\n        \"target_unique_values\": len(set(targets)),\n        \"top_targets\": Counter(targets).most_common(10),\n    }\n\n\ndef analyze_unit_family(rows: list[dict[str, str]]) -> dict[str, Any]:\n    slopes = []\n    query_values = []\n    for row in rows:\n        prompt_slopes = []\n        for line in row[\"prompt\"].splitlines():\n            match = re.match(r\"([0-9.]+) m becomes ([0-9.]+)\", line)\n            if match:\n                left, right = map(float, match.groups())\n                prompt_slopes.append(right / left)\n            query_match = re.search(r\"convert the following measurement: ([0-9.]+) m\", line)\n            if query_match:\n                query_values.append(float(query_match.group(1)))\n        if prompt_slopes:\n            slopes.append(sum(prompt_slopes) / len(prompt_slopes))\n    return {\n        \"slope_min\": min(slopes),\n        \"slope_median\": statistics.median(slopes),\n        \"slope_max\": max(slopes),\n        \"query_value_min\": min(query_values),\n        \"query_value_max\": max(query_values),\n    }\n\n\ndef analyze_gravity_family(rows: list[dict[str, str]]) -> dict[str, Any]:\n    g_values = []\n    query_times = []\n    for row in rows:\n        inferred = []\n        for line in row[\"prompt\"].splitlines():\n            match = re.match(r\"For t = ([0-9.]+)s, distance = ([0-9.]+) m\", line)\n            if match:\n                time_value, distance = map(float, match.groups())\n                inferred.append((2.0 * distance) / (time_value * time_value))\n            query_match = re.search(r\"for t = ([0-9.]+)s given d\", line, flags=re.IGNORECASE)\n            if query_match:\n                query_times.append(float(query_match.group(1)))\n        if inferred:\n            g_values.append(sum(inferred) / len(inferred))\n    return {\n        \"g_min\": min(g_values),\n        \"g_median\": statistics.median(g_values),\n        \"g_max\": max(g_values),\n        \"query_time_min\": min(query_times),\n        \"query_time_max\": max(query_times),\n    }\n\n\ndef analyze_equation_family(rows: list[dict[str, str]]) -> dict[str, Any]:\n    subfamilies = Counter()\n    numeric_operator_counter = Counter()\n    integer_answers = 0\n    negative_integer_answers = 0\n    leading_zero_integer_answers = 0\n    for row in rows:\n        query = row[\"prompt\"].splitlines()[-1].split(\": \", 1)[1]\n        match = NUMERIC_EXPR_RE.fullmatch(query)\n        if match:\n            subfamilies[\"numeric_expression\"] += 1\n            numeric_operator_counter[match.group(1)] += 1\n        else:\n            subfamilies[\"symbol_string\"] += 1\n        if INTEGER_RE.fullmatch(row[\"answer\"]):\n            integer_answers += 1\n            if row[\"answer\"].startswith(\"-\"):\n                negative_integer_answers += 1\n            if len(row[\"answer\"]) > 1 and row[\"answer\"].startswith(\"0\"):\n                leading_zero_integer_answers += 1\n    return {\n        \"subfamilies\": dict(subfamilies),\n        \"numeric_query_operators\": numeric_operator_counter.most_common(),\n        \"integer_answers\": integer_answers,\n        \"negative_integer_answers\": negative_integer_answers,\n        \"leading_zero_integer_answers\": leading_zero_integer_answers,\n    }\n\n\ndef build_profile(train_rows: list[dict[str, str]], test_rows: list[dict[str, str]]) -> dict[str, Any]:\n    family_rows: dict[str, list[dict[str, str]]] = defaultdict(list)\n    family_labels: dict[str, str] = {}\n    family_first_lines: dict[str, str] = {}\n    for row in train_rows:\n        family_id, label = classify_family(row[\"prompt\"])\n        family_rows[family_id].append(row)\n        family_labels[family_id] = label\n        family_first_lines[family_id] = row[\"prompt\"].splitlines()[0]\n\n    overall = {\n        \"train_rows\": len(train_rows),\n        \"test_rows\": len(test_rows),\n        \"exact_test_prompt_overlap_with_train\": sum(\n            1 for row in test_rows if any(row[\"id\"] == train[\"id\"] and row[\"prompt\"] == train[\"prompt\"] for train in train_rows)\n        ),\n        \"prompt_char_lengths\": summarize_lengths([len(row[\"prompt\"]) for row in train_rows]),\n        \"answer_char_lengths\": summarize_lengths([len(row[\"answer\"]) for row in train_rows]),\n        \"prompt_word_lengths\": summarize_lengths([len(row[\"prompt\"].split()) for row in train_rows]),\n        \"answer_word_lengths\": summarize_lengths([len(row[\"answer\"].split()) for row in train_rows]),\n        \"answer_schema_counts\": dict(Counter(classify_answer(row[\"answer\"]) for row in train_rows)),\n    }\n\n    family_summary: dict[str, Any] = {}\n    for family_id, rows in family_rows.items():\n        family_summary[family_id] = {\n            \"label\": family_labels[family_id],\n            \"first_line\": family_first_lines[family_id],\n            \"count\": len(rows),\n            \"share\": len(rows) / len(train_rows),\n            \"prompt_char_lengths\": summarize_lengths([len(row[\"prompt\"]) for row in rows]),\n            \"answer_char_lengths\": summarize_lengths([len(row[\"answer\"]) for row in rows]),\n            \"example_count_distribution\": dict(Counter(example_count(row[\"prompt\"], family_id) for row in rows)),\n            \"answer_schema_counts\": dict(Counter(classify_answer(row[\"answer\"]) for row in rows)),\n            \"sample_ids\": [row[\"id\"] for row in rows[:3]],\n        }\n\n    family_summary[\"text_cipher\"][\"specialized\"] = analyze_text_family(family_rows[\"text_cipher\"])\n    family_summary[\"roman_numeral\"][\"specialized\"] = analyze_roman_family(family_rows[\"roman_numeral\"])\n    family_summary[\"unit_conversion\"][\"specialized\"] = analyze_unit_family(family_rows[\"unit_conversion\"])\n    family_summary[\"gravity\"][\"specialized\"] = analyze_gravity_family(family_rows[\"gravity\"])\n    family_summary[\"equation_transform\"][\"specialized\"] = analyze_equation_family(family_rows[\"equation_transform\"])\n\n    return {\n        \"overall\": overall,\n        \"families\": family_summary,\n    }\n\n\ndef render_report(profile: dict[str, Any]) -> str:\n    lines: list[str] = []\n    overall = profile[\"overall\"]\n    lines.append(\"Wonderland benchmark profile\")\n    lines.append(\"=\" * 28)\n    lines.append(f\"Train rows: {overall['train_rows']}\")\n    lines.append(f\"Visible test rows: {overall['test_rows']}\")\n    lines.append(\n        f\"Visible test rows that exactly overlap train: {overall['exact_test_prompt_overlap_with_train']}\"\n    )\n    lines.append(\"\")\n    lines.append(\"Overall length statistics\")\n    lines.append(\"-\" * 24)\n    for key in [\"prompt_char_lengths\", \"answer_char_lengths\", \"prompt_word_lengths\", \"answer_word_lengths\"]:\n        lines.append(f\"{key}: {overall[key]}\")\n    lines.append(f\"answer_schema_counts: {overall['answer_schema_counts']}\")\n    lines.append(\"\")\n    lines.append(\"Family summary\")\n    lines.append(\"-\" * 14)\n    for family_id, family in sorted(profile[\"families\"].items(), key=lambda item: (-item[1][\"count\"], item[0])):\n        lines.append(f\"{family_id}: {family['count']} rows ({family['share']:.1%})\")\n        lines.append(f\"  marker: {family['first_line']}\")\n        lines.append(f\"  example_count_distribution: {family['example_count_distribution']}\")\n        lines.append(f\"  answer_schema_counts: {family['answer_schema_counts']}\")\n        lines.append(f\"  prompt_char_lengths: {family['prompt_char_lengths']}\")\n        lines.append(f\"  answer_char_lengths: {family['answer_char_lengths']}\")\n        if \"specialized\" in family:\n            lines.append(f\"  specialized: {family['specialized']}\")\n    return \"\\n\".join(lines)\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--json\", action=\"store_true\", help=\"emit JSON instead of a text report\")\n    return parser.parse_args()\n\n\ndef main() -> None:\n    args = parse_args()\n    train_rows = read_csv(TRAIN_PATH)\n    test_rows = read_csv(TEST_PATH)\n    profile = build_profile(train_rows, test_rows)\n    if args.json:\n        print(json.dumps(profile, indent=2, sort_keys=True))\n    else:\n        print(render_report(profile))\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "scripts/build_router.py": "#!/usr/bin/env python3\n\"\"\"Build and evaluate a benchmark-specific router for the Wonderland reasoning suite.\n\nThe router is intentionally hand-crafted from dataset forensics rather than learned from\nsurface labels. It combines lexical markers with structural checks so that routing does\nnot depend on a single brittle keyword.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport math\nimport re\nfrom collections import Counter, defaultdict\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom statistics import mean\nfrom typing import Dict, Iterable, List, Optional, Sequence, Tuple\n\nFAMILY_ORDER: Sequence[str] = (\n    \"bit_transform\",\n    \"text_cipher\",\n    \"roman_numeral\",\n    \"unit_conversion\",\n    \"gravity\",\n    \"equation_transform\",\n)\n\nTOP_LEVEL_MARKERS: Dict[str, Sequence[str]] = {\n    \"bit_transform\": (\n        \"8-bit binary numbers\",\n        \"input -> output\",\n        \"bit manipulation rule\",\n    ),\n    \"text_cipher\": (\n        \"secret encryption rules are used on text\",\n        \"decrypt the following text\",\n        \"here are some examples:\",\n    ),\n    \"roman_numeral\": (\n        \"different numeral system\",\n        \"write the number\",\n        \"wonderland numeral system\",\n    ),\n    \"unit_conversion\": (\n        \"secret unit conversion\",\n        \"convert the following measurement\",\n        \"becomes\",\n    ),\n    \"gravity\": (\n        \"gravitational constant has been secretly changed\",\n        \"d = 0.5*g*t^2\",\n        \"for t =\",\n    ),\n    \"equation_transform\": (\n        \"transformation rules is applied to equations\",\n        \"transformation rules are applied to equations\",\n        \"determine the result for:\",\n    ),\n}\n\nROMAN_NUMERAL_RE = re.compile(r\"\\b([1-9][0-9]?)\\s*->\\s*([IVXLCDM]+)\\b\")\nUNIT_EXAMPLE_RE = re.compile(r\"([0-9]+(?:\\.[0-9]+)?)\\s*m becomes\\s*([0-9]+(?:\\.[0-9]+)?)\")\nGRAVITY_EXAMPLE_RE = re.compile(\n    r\"For t = ([0-9]+(?:\\.[0-9]+)?)s, distance = ([0-9]+(?:\\.[0-9]+)?) m\",\n    re.IGNORECASE,\n)\nBINARY_IO_RE = re.compile(r\"\\b[01]{8}\\b\\s*->\\s*\\b[01]{8}\\b\")\nQUERY_RE = re.compile(r\"Now,\\s*(?:decrypt the following text|write the number .*?|convert the following measurement|determine the output for|determine the result for|determine the falling distance for)\\s*:?\\s*(.+)\", re.IGNORECASE | re.DOTALL)\nNUMERIC_EQUATION_QUERY_RE = re.compile(r\"^[0-9]{2}\\D[0-9]{2}$\")\nPUNCT_QUERY_RE = re.compile(r\"^[\\W_]+$\")\nLOWER_ALPHA_RE = re.compile(r\"^[a-z]+(?: [a-z]+){2,4}$\")\n\n\n@dataclass\nclass RouteResult:\n    family: str\n    confidence: float\n    subfamily: Optional[str]\n    triggers: List[str]\n    score_breakdown: Dict[str, float]\n    ambiguity_flag: bool\n    fallback_policy: str\n    answer_schema: str\n    second_choice: Optional[str]\n    margin: float\n\n\n@dataclass\nclass PromptFeatures:\n    prompt: str\n    prompt_lower: str\n    marker_hits: Dict[str, List[str]]\n    query: str\n    binary_example_count: int\n    roman_example_count: int\n    unit_example_count: int\n    gravity_example_count: int\n    arrow_count: int\n    becomes_count: int\n    has_decrypt_query: bool\n    has_bit_query: bool\n    has_gravity_formula: bool\n    has_equation_query: bool\n    has_roman_instruction: bool\n    has_unit_instruction: bool\n\n\ndef extract_query(prompt: str) -> str:\n    match = QUERY_RE.search(prompt)\n    if not match:\n        return \"\"\n    return match.group(1).strip().rstrip(\".\")\n\n\ndef extract_features(prompt: str) -> PromptFeatures:\n    prompt_lower = prompt.lower()\n    marker_hits: Dict[str, List[str]] = {}\n    for family, markers in TOP_LEVEL_MARKERS.items():\n        hits = [marker for marker in markers if marker in prompt_lower]\n        if hits:\n            marker_hits[family] = hits\n\n    return PromptFeatures(\n        prompt=prompt,\n        prompt_lower=prompt_lower,\n        marker_hits=marker_hits,\n        query=extract_query(prompt),\n        binary_example_count=len(BINARY_IO_RE.findall(prompt)),\n        roman_example_count=len(ROMAN_NUMERAL_RE.findall(prompt)),\n        unit_example_count=len(UNIT_EXAMPLE_RE.findall(prompt)),\n        gravity_example_count=len(GRAVITY_EXAMPLE_RE.findall(prompt)),\n        arrow_count=prompt.count(\"->\"),\n        becomes_count=prompt_lower.count(\" becomes \"),\n        has_decrypt_query=\"decrypt the following text\" in prompt_lower,\n        has_bit_query=\"determine the output for:\" in prompt_lower,\n        has_gravity_formula=\"d = 0.5*g*t^2\" in prompt_lower,\n        has_equation_query=\"determine the result for:\" in prompt_lower,\n        has_roman_instruction=\"write the number\" in prompt_lower,\n        has_unit_instruction=\"convert the following measurement\" in prompt_lower,\n    )\n\n\ndef score_family(features: PromptFeatures, family: str) -> Tuple[float, List[str]]:\n    score = 0.0\n    triggers: List[str] = []\n\n    for marker in features.marker_hits.get(family, []):\n        score += 0.34\n        triggers.append(f\"marker:{marker}\")\n\n    if family == \"bit_transform\":\n        if features.binary_example_count >= 3:\n            score += 0.28\n            triggers.append(f\"structure:{features.binary_example_count}_binary_io_pairs\")\n        if features.has_bit_query and re.search(r\"\\b[01]{8}\\b\", features.query):\n            score += 0.18\n            triggers.append(\"query:8bit_binary_target\")\n        if \"bit shift\" in features.prompt_lower or \"xor\" in features.prompt_lower:\n            score += 0.15\n            triggers.append(\"ops:bitwise_operator_inventory\")\n\n    elif family == \"text_cipher\":\n        if features.has_decrypt_query:\n            score += 0.24\n            triggers.append(\"query:decrypt_text\")\n        if features.arrow_count >= 3 and \" -> \" in features.prompt and re.search(r\"\\b[a-z]{3,}\\b\", features.query.lower()):\n            score += 0.18\n            triggers.append(\"structure:example_cipher_pairs\")\n        if \"secret encryption rules\" in features.prompt_lower:\n            score += 0.15\n            triggers.append(\"theme:text_encryption\")\n\n    elif family == \"roman_numeral\":\n        if features.roman_example_count >= 2:\n            score += 0.28\n            triggers.append(f\"structure:{features.roman_example_count}_roman_examples\")\n        if features.has_roman_instruction and re.search(r\"\\bnumber\\s+[0-9]{1,3}\\b\", features.prompt_lower):\n            score += 0.20\n            triggers.append(\"query:write_number_as_numeral\")\n        if \"numeral system\" in features.prompt_lower:\n            score += 0.18\n            triggers.append(\"theme:numeral_system\")\n\n    elif family == \"unit_conversion\":\n        if features.unit_example_count >= 2:\n            score += 0.28\n            triggers.append(f\"structure:{features.unit_example_count}_unit_pairs\")\n        if features.has_unit_instruction:\n            score += 0.20\n            triggers.append(\"query:convert_measurement\")\n        if features.becomes_count >= 2:\n            score += 0.16\n            triggers.append(f\"pattern:{features.becomes_count}_becomes_lines\")\n\n    elif family == \"gravity\":\n        if features.gravity_example_count >= 2:\n            score += 0.28\n            triggers.append(f\"structure:{features.gravity_example_count}_gravity_observations\")\n        if features.has_gravity_formula:\n            score += 0.22\n            triggers.append(\"formula:d=0.5*g*t^2\")\n        if \"falling distance\" in features.prompt_lower:\n            score += 0.14\n            triggers.append(\"query:falling_distance\")\n\n    elif family == \"equation_transform\":\n        if features.has_equation_query:\n            score += 0.22\n            triggers.append(\"query:determine_equation_result\")\n        if \"examples:\" in features.prompt_lower and \"=\" in features.prompt:\n            score += 0.18\n            triggers.append(\"structure:equation_examples\")\n        if re.search(r\"[`'\\\\/|@#^&!?()\\[\\]{}<>*-]\", features.query):\n            score += 0.18\n            triggers.append(\"query:symbolic_operator_alphabet\")\n        if re.search(r\"`.+? = .+\", features.prompt):\n            score += 0.16\n            triggers.append(\"format:backticked_symbol_examples\")\n\n    return min(score, 1.6), triggers\n\n\ndef infer_subfamily(features: PromptFeatures, family: str) -> Optional[str]:\n    query = features.query.strip()\n    if family == \"equation_transform\":\n        if NUMERIC_EQUATION_QUERY_RE.fullmatch(query):\n            return \"numeric_expression\"\n        if PUNCT_QUERY_RE.fullmatch(query):\n            return \"punctuation_string\"\n        return \"mixed_symbolic\"\n    if family == \"text_cipher\":\n        token_count = len(query.split())\n        if token_count == 3:\n            return \"three_word_clause\"\n        if token_count == 4:\n            return \"four_word_clause\"\n        if token_count == 5:\n            return \"five_word_clause\"\n        return \"other_clause_length\"\n    return None\n\n\ndef answer_schema_for_family(family: str, subfamily: Optional[str]) -> str:\n    if family == \"bit_transform\":\n        return \"exactly 8 binary digits\"\n    if family == \"text_cipher\":\n        return \"3-5 lowercase words separated by single spaces\"\n    if family == \"roman_numeral\":\n        return \"uppercase Roman numeral\"\n    if family == \"unit_conversion\":\n        return \"decimal with exactly 2 places\"\n    if family == \"gravity\":\n        return \"decimal matching inferred benchmark precision\"\n    if family == \"equation_transform\":\n        if subfamily == \"numeric_expression\":\n            return \"short symbolic or integer string; preserve leading zeros/minus sign\"\n        return \"symbolic punctuation string; preserve exact characters\"\n    return \"unconstrained\"\n\n\ndef route_prompt(prompt: str) -> RouteResult:\n    features = extract_features(prompt)\n    score_breakdown: Dict[str, float] = {}\n    trigger_map: Dict[str, List[str]] = {}\n    for family in FAMILY_ORDER:\n        score, triggers = score_family(features, family)\n        score_breakdown[family] = score\n        trigger_map[family] = triggers\n\n    ranked = sorted(score_breakdown.items(), key=lambda item: item[1], reverse=True)\n    top_family, top_score = ranked[0]\n    second_family, second_score = ranked[1]\n    margin = top_score - second_score\n\n    raw_confidence = 0.45 + 0.35 * min(top_score / 1.2, 1.0) + 0.20 * min(max(margin, 0.0) / 0.6, 1.0)\n    confidence = max(0.0, min(raw_confidence, 0.99))\n    ambiguity_flag = confidence < 0.72 or margin < 0.18\n\n    if top_score < 0.55:\n        fallback_policy = \"send to general symbolic fallback prompt and log for taxonomy review\"\n    elif ambiguity_flag:\n        fallback_policy = (\n            f\"run {top_family} solver first, then backoff to {second_family} solver if schema validation fails\"\n        )\n    else:\n        fallback_policy = f\"run {top_family} solver only; if output schema check fails, use generic benchmark fallback\"\n\n    subfamily = infer_subfamily(features, top_family)\n    return RouteResult(\n        family=top_family,\n        confidence=round(confidence, 4),\n        subfamily=subfamily,\n        triggers=trigger_map[top_family],\n        score_breakdown={k: round(v, 3) for k, v in score_breakdown.items()},\n        ambiguity_flag=ambiguity_flag,\n        fallback_policy=fallback_policy,\n        answer_schema=answer_schema_for_family(top_family, subfamily),\n        second_choice=second_family,\n        margin=round(margin, 4),\n    )\n\n\ndef infer_gold_family(prompt: str) -> str:\n    prompt_lower = prompt.lower()\n    if \"8-bit binary numbers\" in prompt_lower:\n        return \"bit_transform\"\n    if \"decrypt the following text\" in prompt_lower:\n        return \"text_cipher\"\n    if \"write the number\" in prompt_lower and \"numeral system\" in prompt_lower:\n        return \"roman_numeral\"\n    if \"convert the following measurement\" in prompt_lower:\n        return \"unit_conversion\"\n    if \"d = 0.5*g*t^2\" in prompt_lower:\n        return \"gravity\"\n    if \"determine the result for:\" in prompt_lower:\n        return \"equation_transform\"\n    raise ValueError(\"Unable to infer gold family from prompt template\")\n\n\ndef infer_gold_subfamily(prompt: str, family: str) -> Optional[str]:\n    if family != \"equation_transform\":\n        return None\n    query = extract_query(prompt)\n    if NUMERIC_EQUATION_QUERY_RE.fullmatch(query):\n        return \"numeric_expression\"\n    if PUNCT_QUERY_RE.fullmatch(query):\n        return \"punctuation_string\"\n    return \"mixed_symbolic\"\n\n\ndef load_rows(csv_path: Path) -> List[Dict[str, str]]:\n    with csv_path.open(newline=\"\") as handle:\n        return list(csv.DictReader(handle))\n\n\ndef evaluate_router(rows: Iterable[Dict[str, str]]) -> Dict[str, object]:\n    total = 0\n    correct = 0\n    ambiguous = 0\n    family_counts = Counter()\n    family_correct = Counter()\n    confidence_by_family: Dict[str, List[float]] = defaultdict(list)\n    eq_total = 0\n    eq_correct = 0\n    examples: List[Tuple[str, str, str, float, List[str]]] = []\n\n    for row in rows:\n        prompt = row[\"prompt\"]\n        gold_family = infer_gold_family(prompt)\n        routed = route_prompt(prompt)\n        total += 1\n        family_counts[gold_family] += 1\n        confidence_by_family[gold_family].append(routed.confidence)\n        if routed.ambiguity_flag:\n            ambiguous += 1\n        if routed.family == gold_family:\n            correct += 1\n            family_correct[gold_family] += 1\n        elif len(examples) < 10:\n            examples.append((row.get(\"id\", \"?\"), gold_family, routed.family, routed.confidence, routed.triggers))\n\n        if gold_family == \"equation_transform\":\n            eq_total += 1\n            if routed.subfamily == infer_gold_subfamily(prompt, gold_family):\n                eq_correct += 1\n\n    summary = {\n        \"rows\": total,\n        \"top_level_accuracy\": (correct / total) if total else math.nan,\n        \"ambiguous_share\": (ambiguous / total) if total else math.nan,\n        \"family_accuracy\": {\n            family: {\n                \"count\": family_counts[family],\n                \"accuracy\": (family_correct[family] / family_counts[family]) if family_counts[family] else math.nan,\n                \"mean_confidence\": mean(confidence_by_family[family]) if confidence_by_family[family] else math.nan,\n            }\n            for family in FAMILY_ORDER\n        },\n        \"equation_subfamily_accuracy\": (eq_correct / eq_total) if eq_total else math.nan,\n        \"misroutes\": examples,\n    }\n    return summary\n\n\ndef print_evaluation(summary: Dict[str, object]) -> None:\n    print(\"=== Router evaluation ===\")\n    print(f\"Rows: {summary['rows']}\")\n    print(f\"Top-level routing accuracy: {summary['top_level_accuracy']:.4f}\")\n    print(f\"Ambiguity-flag share: {summary['ambiguous_share']:.4f}\")\n    print(f\"Equation subfamily accuracy: {summary['equation_subfamily_accuracy']:.4f}\")\n    print(\"\\nPer-family metrics:\")\n    family_accuracy: Dict[str, Dict[str, float]] = summary[\"family_accuracy\"]  # type: ignore[assignment]\n    for family in FAMILY_ORDER:\n        stats = family_accuracy[family]\n        print(\n            f\"  - {family:<18} count={stats['count']:>4} \"\n            f\"accuracy={stats['accuracy']:.4f} mean_confidence={stats['mean_confidence']:.4f}\"\n        )\n\n    misroutes: List[Tuple[str, str, str, float, List[str]]] = summary[\"misroutes\"]  # type: ignore[assignment]\n    if misroutes:\n        print(\"\\nSample misroutes:\")\n        for row_id, gold, pred, conf, triggers in misroutes:\n            joined = \", \".join(triggers)\n            print(f\"  - id={row_id} gold={gold} pred={pred} confidence={conf:.4f} triggers=[{joined}]\")\n    else:\n        print(\"\\nNo misroutes found on the evaluated split.\")\n\n\ndef demo_routes(rows: Sequence[Dict[str, str]], limit: int) -> None:\n    print(\"\\n=== Sample routes ===\")\n    for row in rows[:limit]:\n        routed = route_prompt(row[\"prompt\"])\n        print(f\"id={row.get('id', '?')} family={routed.family} subfamily={routed.subfamily} confidence={routed.confidence:.4f}\")\n        print(f\"  triggers: {', '.join(routed.triggers)}\")\n        print(f\"  fallback: {routed.fallback_policy}\")\n        print(f\"  answer schema: {routed.answer_schema}\")\n\n\nNORMALIZATION_RULES = {\n    \"bit_transform\": \"Emit only the final 8-bit string; left-pad with zeros to width 8 if solver returns fewer bits.\",\n    \"text_cipher\": \"Lowercase all tokens, collapse repeated whitespace, and reject punctuation outside apostrophe-free words.\",\n    \"roman_numeral\": \"Emit uppercase Roman numeral letters only; no surrounding prose.\",\n    \"unit_conversion\": \"Round to exactly two decimals using decimal arithmetic and always print two digits after the point.\",\n    \"gravity\": \"Infer the prompt precision from demonstrations, round consistently, and avoid trimming required trailing zeros.\",\n    \"equation_transform\": \"Preserve exact symbolic characters, leading zeros, minus signs, and output length; never paraphrase or add spaces.\",\n}\n\n\ndef print_normalization_rules() -> None:\n    print(\"\\n=== Final answer normalization policy ===\")\n    for family in FAMILY_ORDER:\n        print(f\"  - {family}: {NORMALIZATION_RULES[family]}\")\n    print(\"  - final emission: wrap the normalized answer as \\\\boxed{answer} and avoid extra reasoning after the box.\")\n\n\nif __name__ == \"__main__\":\n    parser = argparse.ArgumentParser(description=\"Build and evaluate the Wonderland task-family router.\")\n    parser.add_argument(\"--csv\", type=Path, default=Path(\"train.csv\"), help=\"CSV file to evaluate. Defaults to train.csv\")\n    parser.add_argument(\"--demo-limit\", type=int, default=5, help=\"Number of example routes to print\")\n    parser.add_argument(\"--skip-demo\", action=\"store_true\", help=\"Skip sample route printing\")\n    args = parser.parse_args()\n\n    rows = load_rows(args.csv)\n    summary = evaluate_router(rows)\n    print_evaluation(summary)\n    if not args.skip_demo:\n        demo_routes(rows, args.demo_limit)\n    print_normalization_rules()\n",
  "scripts/eval_baselines.py": "#!/usr/bin/env python3\n\"\"\"Evaluate lightweight offline baselines for the Wonderland benchmark.\n\nThe goal is not to solve every family. The goal is to establish a reproducible,\nhonest reference point across both random and structure-aware validation splits.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport math\nimport re\nimport statistics\nfrom collections import Counter, defaultdict\nfrom dataclasses import dataclass\nfrom decimal import Decimal, ROUND_HALF_UP, getcontext\nfrom pathlib import Path\nfrom typing import Callable, Iterable, Sequence\n\ngetcontext().prec = 28\n\nROOT = Path(__file__).resolve().parents[1]\nTRAIN_PATH = ROOT / \"train.csv\"\n\nFAMILIES: Sequence[str] = (\n    \"bit_transform\",\n    \"text_cipher\",\n    \"roman_numeral\",\n    \"unit_conversion\",\n    \"gravity\",\n    \"equation_transform\",\n)\n\n\n@dataclass(frozen=True)\nclass RowRecord:\n    row_id: str\n    prompt: str\n    answer: str\n    family: str\n    subfamily: str | None\n    structure_signature: str\n\n\ndef read_rows(path: Path) -> list[dict[str, str]]:\n    with path.open(newline=\"\", encoding=\"utf-8\") as handle:\n        return list(csv.DictReader(handle))\n\n\ndef classify_family(prompt: str) -> str:\n    if \"8-bit binary numbers\" in prompt:\n        return \"bit_transform\"\n    if \"secret encryption rules are used on text\" in prompt:\n        return \"text_cipher\"\n    if \"different numeral system\" in prompt:\n        return \"roman_numeral\"\n    if \"secret unit conversion\" in prompt:\n        return \"unit_conversion\"\n    if \"gravitational constant has been secretly changed\" in prompt:\n        return \"gravity\"\n    if \"transformation rules are applied to equations\" in prompt or \"transformation rules is applied to equations\" in prompt:\n        return \"equation_transform\"\n    raise ValueError(f\"Unknown family for prompt: {prompt[:80]!r}\")\n\n\ndef extract_query(prompt: str, prefix: str) -> str:\n    match = re.search(re.escape(prefix) + r\"\\s*(.+)\", prompt, flags=re.IGNORECASE | re.DOTALL)\n    return match.group(1).strip() if match else \"\"\n\n\ndef stable_fold(key: str, num_folds: int) -> int:\n    digest = hashlib.md5(key.encode(\"utf-8\")).hexdigest()\n    return int(digest[:8], 16) % num_folds\n\n\ndef decimal_quantize(value: Decimal, places: int) -> str:\n    quantum = Decimal(\"1\") if places == 0 else Decimal(\"1.\" + (\"0\" * places))\n    return format(value.quantize(quantum, rounding=ROUND_HALF_UP), f\".{places}f\")\n\n\ndef roman_encode(number: int) -> str:\n    pairs = (\n        (100, \"C\"),\n        (90, \"XC\"),\n        (50, \"L\"),\n        (40, \"XL\"),\n        (10, \"X\"),\n        (9, \"IX\"),\n        (5, \"V\"),\n        (4, \"IV\"),\n        (1, \"I\"),\n    )\n    output: list[str] = []\n    remainder = number\n    for value, token in pairs:\n        while remainder >= value:\n            output.append(token)\n            remainder -= value\n    return \"\".join(output)\n\n\ndef gravity_precision(distances: Sequence[str], value: Decimal) -> int:\n    if any(distance.endswith(\".0\") or re.fullmatch(r\"\\d+\\.\\d\", distance) for distance in distances):\n        rounded_2 = decimal_quantize(value, 2)\n        if rounded_2.endswith(\"0\"):\n            return 1\n    return 2\n\n\ndef normalize_prediction(text: str) -> str:\n    box_match = re.search(r\"\\\\boxed\\{([^{}]+)\\}\", text)\n    if box_match:\n        text = box_match.group(1)\n    return text.strip()\n\n\ndef validate_answer_format(family: str, prediction: str) -> bool:\n    patterns = {\n        \"bit_transform\": re.compile(r\"^[01]{8}$\"),\n        \"text_cipher\": re.compile(r\"^[a-z]+(?: [a-z]+){2,4}$\"),\n        \"roman_numeral\": re.compile(r\"^[IVXLCDM]+$\"),\n        \"unit_conversion\": re.compile(r\"^-?\\d+\\.\\d{2}$\"),\n        \"gravity\": re.compile(r\"^-?\\d+\\.\\d{1,2}$\"),\n        \"equation_transform\": re.compile(r\"^\\S+$\"),\n    }\n    return bool(patterns[family].fullmatch(prediction))\n\n\ndef build_structure_signature(prompt: str, family: str, answer: str) -> tuple[str, str | None]:\n    if family == \"roman_numeral\":\n        query = int(re.search(r\"write the number (\\d+)\", prompt).group(1))\n        subfamily = None\n        signature = f\"roman|decade={query//10}|boundary={int(query in {4,9,14,19,39,40,44,49,90,94,99,100})}|examples={prompt.count('->')}\"\n        return signature, subfamily\n    if family == \"unit_conversion\":\n        pairs = [(Decimal(m.group(1)), Decimal(m.group(2))) for m in re.finditer(r\"([0-9.]+) m becomes ([0-9.]+)\", prompt)]\n        slope = statistics.median(float(y / x) for x, y in pairs)\n        query = float(re.search(r\"measurement: ([0-9.]+) m\", prompt).group(1))\n        subfamily = None\n        signature = f\"unit|slope_bin={int(slope*10)}|query_bin={int(query//5)}|examples={len(pairs)}\"\n        return signature, subfamily\n    if family == \"gravity\":\n        triples = [(Decimal(m.group(1)), Decimal(m.group(2))) for m in re.finditer(r\"For t = ([0-9.]+)s, distance = ([0-9.]+) m\", prompt)]\n        g_value = statistics.median(float((Decimal('2') * d) / (t * t)) for t, d in triples)\n        precisions = sorted({len(m.group(2).split('.', 1)[1]) if '.' in m.group(2) else 0 for m in re.finditer(r\"For t = ([0-9.]+)s, distance = ([0-9.]+) m\", prompt)})\n        query_t = float(re.search(r\"for t = ([0-9.]+)s given d\", prompt, flags=re.IGNORECASE).group(1))\n        subfamily = None\n        signature = f\"gravity|g_bin={int(g_value)}|query_bin={int(query_t)}|prec={'-'.join(map(str, precisions))}|examples={len(triples)}\"\n        return signature, subfamily\n    if family == \"text_cipher\":\n        query = extract_query(prompt, \"Now, decrypt the following text:\")\n        token_count = len(query.split())\n        subfamily = {3: \"three_word_clause\", 4: \"four_word_clause\", 5: \"five_word_clause\"}.get(len(answer.split()), \"other\")\n        signature = f\"text|tokens={token_count}|answer_len={len(answer.split())}|examples={prompt.count('->')}\"\n        return signature, subfamily\n    if family == \"bit_transform\":\n        query = extract_query(prompt, \"Now, determine the output for:\")\n        hamming_weight = query.count(\"1\")\n        subfamily = None\n        signature = f\"bit|examples={prompt.count('->')}|hw_bin={hamming_weight//2}\"\n        return signature, subfamily\n    query = extract_query(prompt, \"Now, determine the result for:\")\n    numeric_match = re.fullmatch(r\"[0-9]{2}(\\D)[0-9]{2}\", query)\n    subfamily = \"numeric_expression\" if numeric_match else \"symbol_string\"\n    operator = numeric_match.group(1) if numeric_match else f\"len{len(query)}\"\n    answer_kind = \"integer\" if re.fullmatch(r\"-?\\d+\", answer) else \"symbolic\"\n    signature = f\"equation|sub={subfamily}|op={operator}|answer={answer_kind}|examples={prompt.count('=')}\"\n    return signature, subfamily\n\n\ndef build_records(rows: Iterable[dict[str, str]]) -> list[RowRecord]:\n    records: list[RowRecord] = []\n    for row in rows:\n        family = classify_family(row[\"prompt\"])\n        signature, subfamily = build_structure_signature(row[\"prompt\"], family, row[\"answer\"])\n        records.append(\n            RowRecord(\n                row_id=row[\"id\"],\n                prompt=row[\"prompt\"],\n                answer=row[\"answer\"],\n                family=family,\n                subfamily=subfamily,\n                structure_signature=signature,\n            )\n        )\n    return records\n\n\ndef baseline_last_demo(record: RowRecord) -> str:\n    lines = [line.strip() for line in record.prompt.splitlines() if line.strip()]\n    if record.family in {\"bit_transform\", \"text_cipher\", \"roman_numeral\"}:\n        example_lines = [line for line in lines if \" -> \" in line]\n        return example_lines[-1].split(\" -> \", 1)[1] if example_lines else \"\"\n    if record.family == \"unit_conversion\":\n        example_lines = [line for line in lines if \" becomes \" in line]\n        return example_lines[-1].split(\" becomes \", 1)[1] if example_lines else \"\"\n    if record.family == \"gravity\":\n        example_lines = [line for line in lines if line.startswith(\"For t =\")]\n        return example_lines[-1].rsplit(\"=\", 1)[1].replace(\"m\", \"\").strip() if example_lines else \"\"\n    example_lines = [line for line in lines if \" = \" in line]\n    return example_lines[-1].split(\" = \", 1)[1] if example_lines else \"\"\n\n\ndef baseline_solver_lite(record: RowRecord) -> str:\n    prompt = record.prompt\n    if record.family == \"roman_numeral\":\n        value = int(re.search(r\"write the number (\\d+)\", prompt).group(1))\n        return roman_encode(value)\n    if record.family == \"unit_conversion\":\n        pairs = [(Decimal(m.group(1)), Decimal(m.group(2))) for m in re.finditer(r\"([0-9.]+) m becomes ([0-9.]+)\", prompt)]\n        slope = statistics.median(y / x for x, y in pairs)\n        query = Decimal(re.search(r\"measurement: ([0-9.]+) m\", prompt).group(1))\n        return decimal_quantize(query * slope, 2)\n    if record.family == \"gravity\":\n        pairs = [(Decimal(m.group(1)), Decimal(m.group(2))) for m in re.finditer(r\"For t = ([0-9.]+)s, distance = ([0-9.]+) m\", prompt)]\n        inferred_g = statistics.median((Decimal(\"2\") * d) / (t * t) for t, d in pairs)\n        query_t = Decimal(re.search(r\"for t = ([0-9.]+)s given d\", prompt, flags=re.IGNORECASE).group(1))\n        predicted = Decimal(\"0.5\") * inferred_g * query_t * query_t\n        distances = [m.group(2) for m in re.finditer(r\"For t = ([0-9.]+)s, distance = ([0-9.]+) m\", prompt)]\n        places = gravity_precision(distances, predicted)\n        return decimal_quantize(predicted, places)\n    if record.family == \"bit_transform\":\n        query = extract_query(prompt, \"Now, determine the output for:\")\n        return query.zfill(8)[:8]\n    if record.family == \"text_cipher\":\n        example_lines = [line for line in prompt.splitlines() if \" -> \" in line]\n        return example_lines[0].split(\" -> \", 1)[1] if example_lines else \"the cat sees\"\n    example_lines = [line for line in prompt.splitlines() if \" = \" in line]\n    return example_lines[-1].split(\" = \", 1)[1] if example_lines else \"0\"\n\n\nBASELINES: dict[str, Callable[[RowRecord], str]] = {\n    \"last_demo\": baseline_last_demo,\n    \"solver_lite\": baseline_solver_lite,\n}\n\n\ndef evaluate(records: Sequence[RowRecord], fold_assignments: dict[str, int], num_folds: int, baseline_name: str) -> dict[str, object]:\n    predictor = BASELINES[baseline_name]\n    fold_metrics: list[dict[str, object]] = []\n    all_predictions: list[tuple[RowRecord, str]] = []\n    for record in records:\n        prediction = normalize_prediction(predictor(record))\n        all_predictions.append((record, prediction))\n\n    for fold in range(num_folds):\n        subset = [(record, prediction) for record, prediction in all_predictions if fold_assignments[record.row_id] == fold]\n        if not subset:\n            continue\n        correct = sum(prediction == record.answer for record, prediction in subset)\n        format_ok = sum(validate_answer_format(record.family, prediction) for record, prediction in subset)\n        family_total = Counter(record.family for record, _ in subset)\n        family_correct = Counter(record.family for record, prediction in subset if prediction == record.answer)\n        fold_metrics.append(\n            {\n                \"fold\": fold,\n                \"count\": len(subset),\n                \"accuracy\": correct / len(subset),\n                \"format_accuracy\": format_ok / len(subset),\n                \"family_total\": family_total,\n                \"family_correct\": family_correct,\n            }\n        )\n\n    total = len(all_predictions)\n    overall_correct = sum(prediction == record.answer for record, prediction in all_predictions)\n    overall_format = sum(validate_answer_format(record.family, prediction) for record, prediction in all_predictions)\n    family_total = Counter(record.family for record, _ in all_predictions)\n    family_correct = Counter(record.family for record, prediction in all_predictions if prediction == record.answer)\n    return {\n        \"overall_accuracy\": overall_correct / total,\n        \"answer_format_accuracy\": overall_format / total,\n        \"per_family_accuracy\": {family: family_correct[family] / family_total[family] for family in FAMILIES},\n        \"fold_metrics\": fold_metrics,\n    }\n\n\ndef summarize_fold_range(fold_metrics: Sequence[dict[str, object]], key: str) -> tuple[float, float]:\n    values = [metric[key] for metric in fold_metrics]\n    return min(values), max(values)\n\n\ndef print_result(split_name: str, result: dict[str, object]) -> None:\n    low_acc, high_acc = summarize_fold_range(result[\"fold_metrics\"], \"accuracy\")\n    low_fmt, high_fmt = summarize_fold_range(result[\"fold_metrics\"], \"format_accuracy\")\n    print(f\"  {split_name}:\")\n    print(\n        f\"    overall_accuracy={result['overall_accuracy']:.4f} \"\n        f\"fold_range=[{low_acc:.4f}, {high_acc:.4f}]\"\n    )\n    print(\n        f\"    answer_format_accuracy={result['answer_format_accuracy']:.4f} \"\n        f\"fold_range=[{low_fmt:.4f}, {high_fmt:.4f}]\"\n    )\n    print(\"    per_family_accuracy:\")\n    for family in FAMILIES:\n        print(f\"      - {family:<18} {result['per_family_accuracy'][family]:.4f}\")\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--num-folds\", type=int, default=5, help=\"Number of folds per split strategy.\")\n    parser.add_argument(\n        \"--baseline\",\n        choices=tuple(BASELINES),\n        default=\"solver_lite\",\n        help=\"Which baseline predictor to evaluate.\",\n    )\n    return parser.parse_args()\n\n\ndef main() -> None:\n    args = parse_args()\n    records = build_records(read_rows(TRAIN_PATH))\n    random_split = {record.row_id: stable_fold(record.row_id, args.num_folds) for record in records}\n    structure_split = {record.row_id: stable_fold(record.structure_signature, args.num_folds) for record in records}\n\n    random_result = evaluate(records, random_split, args.num_folds, args.baseline)\n    structure_result = evaluate(records, structure_split, args.num_folds, args.baseline)\n\n    print(f\"Baseline: {args.baseline}\")\n    print(f\"Rows evaluated: {len(records)}\")\n    print_result(\"stratified_random_hash\", random_result)\n    print_result(\"structure_aware_hash\", structure_result)\n    drift_gap = random_result[\"overall_accuracy\"] - structure_result[\"overall_accuracy\"]\n    print(f\"  random_vs_structure_accuracy_gap={drift_gap:.4f}\")\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "scripts/generate_synthetic.py": "#!/usr/bin/env python3\n\"\"\"Generate selective synthetic data for high-confidence Wonderland families.\n\nThis script intentionally supports only the currently approved synthetic families:\n- roman_numeral\n- unit_conversion\n- gravity\n\nThe goal is benchmark-faithful augmentation for formatting and parameter coverage,\nnot brute-force volume generation.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport random\nimport re\nfrom dataclasses import dataclass\nfrom decimal import Decimal, ROUND_HALF_UP, getcontext\nfrom pathlib import Path\nfrom typing import Callable, Iterable, Sequence\n\ngetcontext().prec = 28\n\nROOT = Path(__file__).resolve().parents[1]\nTRAIN_PATH = ROOT / \"train.csv\"\nDEFAULT_OUTPUT = ROOT / \"artifacts\" / \"synthetic\" / \"synthetic_train.csv\"\n\nSUPPORTED_FAMILIES = (\"roman_numeral\", \"unit_conversion\", \"gravity\")\nROMAN_BOUNDARIES = {4, 9, 14, 19, 39, 40, 44, 49, 90, 94, 99, 100}\n\n\n@dataclass(frozen=True)\nclass SyntheticRow:\n    row_id: str\n    family: str\n    synthetic_tier: str\n    prompt: str\n    answer: str\n\n\ndef stable_id(prefix: str, index: int, seed: int) -> str:\n    digest = hashlib.md5(f\"{prefix}:{index}:{seed}\".encode(\"utf-8\")).hexdigest()\n    return f\"syn_{digest[:12]}\"\n\n\ndef read_existing_prompts(path: Path) -> set[str]:\n    with path.open(newline=\"\", encoding=\"utf-8\") as handle:\n        return {row[\"prompt\"] for row in csv.DictReader(handle)}\n\n\ndef decimal_places(text: str) -> int:\n    return len(text.split(\".\", 1)[1]) if \".\" in text else 0\n\n\ndef quantize(value: Decimal, places: int) -> str:\n    quantum = Decimal(\"1\") if places == 0 else Decimal(\"1.\" + (\"0\" * places))\n    return format(value.quantize(quantum, rounding=ROUND_HALF_UP), f\".{places}f\")\n\n\ndef roman_encode(number: int) -> str:\n    pairs = (\n        (100, \"C\"),\n        (90, \"XC\"),\n        (50, \"L\"),\n        (40, \"XL\"),\n        (10, \"X\"),\n        (9, \"IX\"),\n        (5, \"V\"),\n        (4, \"IV\"),\n        (1, \"I\"),\n    )\n    output: list[str] = []\n    remaining = number\n    for value, token in pairs:\n        while remaining >= value:\n            output.append(token)\n            remaining -= value\n    return \"\".join(output)\n\n\ndef render_roman_prompt(rng: random.Random, tier: str) -> tuple[str, str]:\n    example_count = rng.randint(3, 5)\n    if tier == \"stress\":\n        query = rng.choice(sorted(ROMAN_BOUNDARIES))\n    else:\n        query = rng.randint(1, 100)\n\n    support_pool = [value for value in range(1, 101) if value != query]\n    if tier == \"stress\":\n        prioritized = [value for value in sorted(ROMAN_BOUNDARIES) if value != query]\n        filler = [value for value in support_pool if value not in ROMAN_BOUNDARIES]\n        rng.shuffle(prioritized)\n        rng.shuffle(filler)\n        support_numbers = prioritized[: example_count - 1]\n        while len(support_numbers) < example_count:\n            support_numbers.append(filler.pop())\n    else:\n        support_numbers = rng.sample(support_pool, example_count)\n\n    examples = \"\\n\".join(f\"{value} -> {roman_encode(value)}\" for value in support_numbers)\n    prompt = (\n        \"In Alice's Wonderland, numbers are secretly converted into a different numeral system. \"\n        \"Some examples are given below:\\n\"\n        f\"{examples}\\n\"\n        f\"Now, write the number {query} in the Wonderland numeral system.\"\n    )\n    return prompt, roman_encode(query)\n\n\ndef sample_decimal(rng: random.Random, minimum: str, maximum: str) -> Decimal:\n    lower = int(Decimal(minimum) * 100)\n    upper = int(Decimal(maximum) * 100)\n    return Decimal(rng.randint(lower, upper)) / Decimal(100)\n\n\ndef render_unit_prompt(rng: random.Random, tier: str) -> tuple[str, str]:\n    example_count = rng.randint(3, 5)\n    if tier == \"stress\":\n        slope = rng.choice(\n            [Decimal(\"0.50\"), Decimal(\"0.51\"), Decimal(\"1.99\"), Decimal(\"2.00\"), Decimal(\"1.25\")]\n        )\n    else:\n        slope = sample_decimal(rng, \"0.50\", \"2.00\")\n\n    used_inputs: set[Decimal] = set()\n\n    def fresh_input() -> Decimal:\n        while True:\n            candidate = sample_decimal(rng, \"5.00\", \"49.99\")\n            if candidate not in used_inputs:\n                used_inputs.add(candidate)\n                return candidate\n\n    examples: list[tuple[Decimal, str]] = []\n    for _ in range(example_count):\n        left = fresh_input()\n        right = quantize(left * slope, 2)\n        examples.append((left, right))\n\n    if tier == \"stress\":\n        query = None\n        for left_int in range(500, 5000):\n            candidate = Decimal(left_int) / Decimal(100)\n            if candidate in used_inputs:\n                continue\n            raw = candidate * slope\n            thousandths = int((raw * 1000) % 10)\n            if thousandths in {4, 5, 6}:\n                query = candidate\n                used_inputs.add(candidate)\n                break\n        if query is None:\n            query = fresh_input()\n    else:\n        query = fresh_input()\n\n    example_lines = \"\\n\".join(f\"{left:.2f} m becomes {right}\" for left, right in examples)\n    answer = quantize(query * slope, 2)\n    prompt = (\n        \"In Alice's Wonderland, a secret unit conversion is applied to measurements. For example:\\n\"\n        f\"{example_lines}\\n\"\n        f\"Now, convert the following measurement: {query:.2f} m\"\n    )\n    return prompt, answer\n\n\ndef infer_precision(example_values: Sequence[str], computed: Decimal, tier: str) -> int:\n    if tier != \"stress\":\n        return 2\n    if any(decimal_places(value) == 1 for value in example_values):\n        rounded_2 = quantize(computed, 2)\n        if rounded_2.endswith(\"0\"):\n            return 1\n    return 2\n\n\ndef render_gravity_prompt(rng: random.Random, tier: str) -> tuple[str, str]:\n    example_count = rng.randint(3, 5)\n    if tier == \"stress\":\n        g = rng.choice(\n            [Decimal(\"4.90\"), Decimal(\"5.00\"), Decimal(\"9.81\"), Decimal(\"12.50\"), Decimal(\"19.60\")]\n        )\n    else:\n        g = sample_decimal(rng, \"4.90\", \"19.60\")\n\n    used_times: set[Decimal] = set()\n\n    def fresh_time() -> Decimal:\n        while True:\n            candidate = Decimal(rng.randint(100, 500)) / Decimal(100)\n            if candidate not in used_times:\n                used_times.add(candidate)\n                return candidate\n\n    example_lines: list[str] = []\n    example_distances: list[str] = []\n    for index in range(example_count):\n        time_value = fresh_time()\n        raw_distance = Decimal(\"0.5\") * g * time_value * time_value\n        if tier == \"stress\" and index == 0:\n            rendered_distance = quantize(raw_distance, 1)\n        else:\n            rendered_distance = quantize(raw_distance, 2)\n        example_distances.append(rendered_distance)\n        time_text = f\"{time_value:.2f}\".rstrip(\"0\").rstrip(\".\")\n        example_lines.append(f\"For t = {time_text}s, distance = {rendered_distance} m\")\n\n    query_time = fresh_time()\n    raw_answer = Decimal(\"0.5\") * g * query_time * query_time\n    precision = infer_precision(example_distances, raw_answer, tier)\n    answer = quantize(raw_answer, precision)\n    query_time_text = f\"{query_time:.2f}\".rstrip(\"0\").rstrip(\".\")\n    rendered_examples = \"\\n\".join(example_lines)\n    prompt = (\n        \"In Alice's Wonderland, the gravitational constant has been secretly changed. Here are some example observations:\\n\"\n        f\"{rendered_examples}\\n\"\n        f\"Now, determine the falling distance for t = {query_time_text}s given d = 0.5*g*t^2.\"\n    )\n    return prompt, answer\n\n\nGENERATOR_MAP: dict[str, Callable[[random.Random, str], tuple[str, str]]] = {\n    \"roman_numeral\": render_roman_prompt,\n    \"unit_conversion\": render_unit_prompt,\n    \"gravity\": render_gravity_prompt,\n}\n\n\ndef answer_matches_schema(family: str, answer: str) -> bool:\n    patterns = {\n        \"roman_numeral\": re.compile(r\"^[IVXLCDM]+$\"),\n        \"unit_conversion\": re.compile(r\"^-?\\d+\\.\\d{2}$\"),\n        \"gravity\": re.compile(r\"^-?\\d+\\.\\d{1,2}$\"),\n    }\n    return bool(patterns[family].fullmatch(answer))\n\n\ndef generate_rows(\n    family: str,\n    count: int,\n    seed: int,\n    tier: str,\n    existing_prompts: set[str],\n) -> list[SyntheticRow]:\n    rng = random.Random(seed)\n    generator = GENERATOR_MAP[family]\n    rows: list[SyntheticRow] = []\n    seen_prompts: set[str] = set(existing_prompts)\n    attempts = 0\n    target = count\n    while len(rows) < target:\n        attempts += 1\n        if attempts > target * 50:\n            raise RuntimeError(f\"Could not generate enough unique rows for {family} ({len(rows)}/{target}).\")\n        prompt, answer = generator(rng, tier)\n        if prompt in seen_prompts:\n            continue\n        if not answer_matches_schema(family, answer):\n            continue\n        seen_prompts.add(prompt)\n        rows.append(\n            SyntheticRow(\n                row_id=stable_id(family, len(rows), seed),\n                family=family,\n                synthetic_tier=tier,\n                prompt=prompt,\n                answer=answer,\n            )\n        )\n    return rows\n\n\ndef preset_counts(preset: str) -> dict[str, int]:\n    if preset == \"mvp\":\n        return {\"roman_numeral\": 600, \"unit_conversion\": 1200, \"gravity\": 1200}\n    if preset == \"full\":\n        return {\"roman_numeral\": 2000, \"unit_conversion\": 4000, \"gravity\": 4000}\n    raise ValueError(f\"Unknown preset: {preset}\")\n\n\ndef write_rows(path: Path, rows: Iterable[SyntheticRow]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\"w\", newline=\"\", encoding=\"utf-8\") as handle:\n        writer = csv.DictWriter(\n            handle,\n            fieldnames=[\"id\", \"family\", \"synthetic_tier\", \"prompt\", \"answer\"],\n        )\n        writer.writeheader()\n        for row in rows:\n            writer.writerow(\n                {\n                    \"id\": row.row_id,\n                    \"family\": row.family,\n                    \"synthetic_tier\": row.synthetic_tier,\n                    \"prompt\": row.prompt,\n                    \"answer\": row.answer,\n                }\n            )\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\n        \"--preset\",\n        choices=(\"mvp\", \"full\"),\n        default=\"mvp\",\n        help=\"Synthetic curriculum preset to generate.\",\n    )\n    parser.add_argument(\n        \"--families\",\n        nargs=\"+\",\n        choices=SUPPORTED_FAMILIES,\n        default=list(SUPPORTED_FAMILIES),\n        help=\"Approved families to generate.\",\n    )\n    parser.add_argument(\n        \"--output\",\n        type=Path,\n        default=DEFAULT_OUTPUT,\n        help=\"Where to write the generated CSV.\",\n    )\n    parser.add_argument(\"--seed\", type=int, default=7, help=\"Random seed.\")\n    parser.add_argument(\n        \"--stress-fraction\",\n        type=float,\n        default=0.15,\n        help=\"Fraction of each family to sample from the stress tier.\",\n    )\n    return parser.parse_args()\n\n\ndef main() -> None:\n    args = parse_args()\n    counts = preset_counts(args.preset)\n    existing_prompts = read_existing_prompts(TRAIN_PATH)\n    all_rows: list[SyntheticRow] = []\n    for offset, family in enumerate(args.families):\n        total = counts[family]\n        stress_count = int(round(total * args.stress_fraction))\n        core_count = total - stress_count\n        family_seed = args.seed + (offset * 1000)\n        all_rows.extend(generate_rows(family, core_count, family_seed, \"core\", existing_prompts))\n        all_rows.extend(generate_rows(family, stress_count, family_seed + 1, \"stress\", existing_prompts))\n\n    write_rows(args.output, all_rows)\n\n    print(f\"Wrote {len(all_rows)} synthetic rows to {args.output}\")\n    by_family: dict[str, int] = {family: 0 for family in args.families}\n    by_tier: dict[str, int] = {\"core\": 0, \"stress\": 0}\n    for row in all_rows:\n        by_family[row.family] += 1\n        by_tier[row.synthetic_tier] += 1\n    for family, count in by_family.items():\n        print(f\"  - {family}: {count}\")\n    for tier, count in by_tier.items():\n        print(f\"  - {tier}: {count}\")\n\n\nif __name__ == \"__main__\":\n    main()\n",
  "scripts/prepare_sft_data.py": "#!/usr/bin/env python3\n\"\"\"Prepare benchmark-specific SFT data for minimal Nemotron LoRA experiments.\n\nThe benchmark favors terse, exact outputs. This script converts the original\ntraining rows plus optional approved synthetic rows into chat-style JSONL files\nthat emphasize the final answer contract used at inference time.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nfrom collections import Counter\nfrom pathlib import Path\nfrom typing import Iterable, Sequence\n\nROOT = Path(__file__).resolve().parents[1]\nDEFAULT_TRAIN_PATH = ROOT / \"train.csv\"\nDEFAULT_OUTPUT_DIR = ROOT / \"artifacts\" / \"sft\"\n\nFAMILY_PREFIXES: Sequence[tuple[str, str]] = (\n    (\n        \"bit_transform\",\n        \"In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers.\",\n    ),\n    (\n        \"text_cipher\",\n        \"In Alice's Wonderland, secret encryption rules are used on text.\",\n    ),\n    (\n        \"roman_numeral\",\n        \"In Alice's Wonderland, numbers are secretly converted into a different numeral system.\",\n    ),\n    (\n        \"unit_conversion\",\n        \"In Alice's Wonderland, a secret unit conversion is applied to measurements.\",\n    ),\n    (\n        \"gravity\",\n        \"In Alice's Wonderland, the gravitational constant has been secretly changed.\",\n    ),\n    (\n        \"equation_transform\",\n        \"In Alice's Wonderland, a secret set of transformation rules is applied to equations.\",\n    ),\n)\n\nDEFAULT_SYSTEM_PROMPT = (\n    \"You solve Wonderland benchmark prompts. Infer the rule silently and return only the final \"\n    \"answer in a single LaTeX box like \\\\boxed{answer}. Never add explanation, units, or extra text.\"\n)\n\nSCHEMA_HINTS = {\n    \"bit_transform\": \"Return exactly 8 binary digits inside one box.\",\n    \"text_cipher\": \"Return only the decrypted lowercase phrase inside one box.\",\n    \"roman_numeral\": \"Return only the uppercase Roman numeral inside one box.\",\n    \"unit_conversion\": \"Return only the converted decimal with exactly 2 digits after the decimal point inside one box.\",\n    \"gravity\": \"Return only the distance as a decimal using the prompt-consistent precision inside one box.\",\n    \"equation_transform\": \"Return only the exact symbolic or integer result inside one box, preserving minus signs and leading zeros when present.\",\n}\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--train-path\", type=Path, default=DEFAULT_TRAIN_PATH)\n    parser.add_argument(\n        \"--synthetic-path\",\n        dest=\"synthetic_paths\",\n        action=\"append\",\n        type=Path,\n        default=[],\n        help=\"Optional synthetic CSV path(s) with at least prompt and answer columns.\",\n    )\n    parser.add_argument(\"--output-dir\", type=Path, default=DEFAULT_OUTPUT_DIR)\n    parser.add_argument(\n        \"--recipe\",\n        choices=(\"original_sft\", \"mixed_sft\", \"family_conditioned\", \"format_stabilization\"),\n        default=\"mixed_sft\",\n        help=\"Training recipe to materialize.\",\n    )\n    parser.add_argument(\n        \"--family-filter\",\n        nargs=\"*\",\n        help=\"Optional list of family ids to keep. Default keeps all real rows and all provided synthetic rows.\",\n    )\n    parser.add_argument(\n        \"--dev-fraction\",\n        type=float,\n        default=0.1,\n        help=\"Fraction of rows assigned to the dev split using a stable hash.\",\n    )\n    parser.add_argument(\n        \"--box-style\",\n        choices=(\"boxed\", \"plain\"),\n        default=\"boxed\",\n        help=\"Assistant target style. 'boxed' is recommended for final-contract tuning.\",\n    )\n    parser.add_argument(\n        \"--max-synthetic-per-family\",\n        type=int,\n        default=0,\n        help=\"Optional cap per synthetic family after loading. 0 means no cap.\",\n    )\n    return parser.parse_args()\n\n\ndef read_csv_rows(path: Path) -> list[dict[str, str]]:\n    with path.open(newline=\"\", encoding=\"utf-8\") as handle:\n        return list(csv.DictReader(handle))\n\n\ndef classify_family(prompt: str) -> str:\n    for family, prefix in FAMILY_PREFIXES:\n        if prompt.startswith(prefix):\n            return family\n    raise ValueError(f\"Unknown family prefix for prompt: {prompt[:120]!r}\")\n\n\ndef stable_bucket(key: str) -> float:\n    digest = hashlib.md5(key.encode(\"utf-8\")).hexdigest()\n    return int(digest[:8], 16) / 0xFFFFFFFF\n\n\ndef load_real_rows(path: Path) -> list[dict[str, str]]:\n    rows = []\n    for row in read_csv_rows(path):\n        rows.append(\n            {\n                \"id\": row[\"id\"],\n                \"prompt\": row[\"prompt\"],\n                \"answer\": row[\"answer\"],\n                \"family\": classify_family(row[\"prompt\"]),\n                \"source\": \"real\",\n            }\n        )\n    return rows\n\n\ndef load_synthetic_rows(paths: Iterable[Path]) -> list[dict[str, str]]:\n    rows: list[dict[str, str]] = []\n    for path in paths:\n        for index, row in enumerate(read_csv_rows(path)):\n            prompt = row[\"prompt\"]\n            rows.append(\n                {\n                    \"id\": row.get(\"id\") or row.get(\"row_id\") or f\"{path.stem}_{index}\",\n                    \"prompt\": prompt,\n                    \"answer\": row[\"answer\"],\n                    \"family\": row.get(\"family\") or classify_family(prompt),\n                    \"source\": row.get(\"source\") or \"synthetic\",\n                }\n            )\n    return rows\n\n\ndef maybe_cap_synthetic(rows: list[dict[str, str]], cap: int) -> list[dict[str, str]]:\n    if cap <= 0:\n        return rows\n    kept: list[dict[str, str]] = []\n    per_family = Counter()\n    for row in rows:\n        family = row[\"family\"]\n        if per_family[family] >= cap:\n            continue\n        per_family[family] += 1\n        kept.append(row)\n    return kept\n\n\ndef build_user_prompt(row: dict[str, str], recipe: str) -> str:\n    prompt = row[\"prompt\"].strip()\n    family = row[\"family\"]\n    if recipe == \"family_conditioned\":\n        return f\"[family={family}]\\n{prompt}\\n\\nSchema hint: {SCHEMA_HINTS[family]}\"\n    if recipe == \"format_stabilization\":\n        return f\"{prompt}\\n\\nReturn only the boxed final answer. {SCHEMA_HINTS[family]}\"\n    return prompt\n\n\ndef build_system_prompt(row: dict[str, str], recipe: str) -> str:\n    if recipe == \"family_conditioned\":\n        return DEFAULT_SYSTEM_PROMPT + f\" The routed family is {row['family']}.\"\n    if recipe == \"format_stabilization\":\n        return DEFAULT_SYSTEM_PROMPT + \" Prioritize format obedience over explanation.\"\n    return DEFAULT_SYSTEM_PROMPT\n\n\ndef render_target(answer: str, box_style: str) -> str:\n    return f\"\\\\boxed{{{answer}}}\" if box_style == \"boxed\" else answer\n\n\ndef make_record(row: dict[str, str], recipe: str, box_style: str) -> dict[str, object]:\n    return {\n        \"id\": row[\"id\"],\n        \"source\": row[\"source\"],\n        \"family\": row[\"family\"],\n        \"messages\": [\n            {\"role\": \"system\", \"content\": build_system_prompt(row, recipe)},\n            {\"role\": \"user\", \"content\": build_user_prompt(row, recipe)},\n            {\"role\": \"assistant\", \"content\": render_target(row[\"answer\"], box_style)},\n        ],\n        \"metadata\": {\n            \"benchmark\": \"wonderland_reasoning\",\n            \"recipe\": recipe,\n            \"answer\": row[\"answer\"],\n            \"target_format\": box_style,\n        },\n    }\n\n\ndef assign_split(row_id: str, source: str, dev_fraction: float) -> str:\n    if source == \"synthetic\":\n        return \"train\"\n    return \"dev\" if stable_bucket(row_id) < dev_fraction else \"train\"\n\n\ndef write_jsonl(path: Path, records: Iterable[dict[str, object]]) -> int:\n    count = 0\n    with path.open(\"w\", encoding=\"utf-8\") as handle:\n        for record in records:\n            handle.write(json.dumps(record, ensure_ascii=False) + \"\\n\")\n            count += 1\n    return count\n\n\ndef main() -> int:\n    args = parse_args()\n    real_rows = load_real_rows(args.train_path)\n    synthetic_rows = maybe_cap_synthetic(load_synthetic_rows(args.synthetic_paths), args.max_synthetic_per_family)\n\n    rows: list[dict[str, str]] = list(real_rows)\n    if args.recipe in {\"mixed_sft\", \"family_conditioned\", \"format_stabilization\"}:\n        rows.extend(synthetic_rows)\n\n    if args.recipe == \"original_sft\":\n        rows = [row for row in rows if row[\"source\"] == \"real\"]\n\n    if args.family_filter:\n        allowed = set(args.family_filter)\n        rows = [row for row in rows if row[\"family\"] in allowed]\n\n    if not rows:\n        raise SystemExit(\"No rows selected. Check --recipe, --family-filter, or synthetic inputs.\")\n\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    dataset_name = f\"{args.recipe}_{args.box_style}\"\n    train_records: list[dict[str, object]] = []\n    dev_records: list[dict[str, object]] = []\n    source_counter = Counter()\n    family_counter = Counter()\n\n    for row in rows:\n        source_counter[row[\"source\"]] += 1\n        family_counter[row[\"family\"]] += 1\n        split = assign_split(row[\"id\"], row[\"source\"], args.dev_fraction)\n        record = make_record(row, args.recipe, args.box_style)\n        if split == \"dev\":\n            dev_records.append(record)\n        else:\n            train_records.append(record)\n\n    train_path = args.output_dir / f\"{dataset_name}.train.jsonl\"\n    dev_path = args.output_dir / f\"{dataset_name}.dev.jsonl\"\n    summary_path = args.output_dir / f\"{dataset_name}.summary.json\"\n\n    train_count = write_jsonl(train_path, train_records)\n    dev_count = write_jsonl(dev_path, dev_records)\n\n    summary = {\n        \"dataset_name\": dataset_name,\n        \"recipe\": args.recipe,\n        \"box_style\": args.box_style,\n        \"train_rows\": train_count,\n        \"dev_rows\": dev_count,\n        \"sources\": dict(source_counter),\n        \"families\": dict(family_counter),\n        \"synthetic_inputs\": [str(path) for path in args.synthetic_paths],\n        \"family_filter\": args.family_filter or [],\n        \"train_path\": str(train_path),\n        \"dev_path\": str(dev_path),\n        \"system_prompt\": DEFAULT_SYSTEM_PROMPT,\n    }\n    summary_path.write_text(json.dumps(summary, indent=2) + \"\\n\", encoding=\"utf-8\")\n\n    print(json.dumps(summary, indent=2))\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n",
  "scripts/package_lora_submission.py": "#!/usr/bin/env python3\n\"\"\"Validate and package a Nemotron-compatible LoRA submission bundle.\n\nThis script does not train a model. It packages an already produced LoRA adapter\ninto a reproducible submission directory and validates the minimum assumptions\nneeded for the competition handoff.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport shutil\nimport tarfile\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[1]\nDEFAULT_OUTPUT_DIR = ROOT / \"artifacts\" / \"submission\"\nEXPECTED_BASE_MODEL = \"Nemotron-3-Nano-30B\"\nREQUIRED_ADAPTER_FILES = (\"adapter_config.json\",)\nOPTIONAL_WEIGHT_FILES = (\"adapter_model.safetensors\", \"adapter_model.bin\")\n\nINFERENCE_CONTRACT = \"\"\"You solve Wonderland benchmark prompts.\n- Infer the rule silently.\n- Return exactly one final answer.\n- The final non-whitespace text must be a single LaTeX box: \\\\boxed{answer}\n- Do not add explanations, units, apologies, or extra punctuation after the box.\n- Preserve benchmark schema exactly:\n  * bit_transform -> 8 binary digits\n  * text_cipher -> lowercase words separated by single spaces\n  * roman_numeral -> uppercase Roman numeral\n  * unit_conversion -> decimal with exactly 2 digits after the decimal point\n  * gravity -> decimal using prompt-consistent precision\n  * equation_transform -> exact symbolic/integer string, preserving leading zeros and minus signs\n\"\"\"\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--adapter-dir\", type=Path, required=True)\n    parser.add_argument(\"--output-dir\", type=Path, default=DEFAULT_OUTPUT_DIR)\n    parser.add_argument(\"--submission-name\", default=\"wonderland_nemotron_lora\")\n    parser.add_argument(\n        \"--expected-base-model\",\n        default=EXPECTED_BASE_MODEL,\n        help=\"Substring that must appear in adapter_config.json base_model_name_or_path.\",\n    )\n    parser.add_argument(\n        \"--allow-missing-weight-file\",\n        action=\"store_true\",\n        help=\"Allow packaging config-only dry runs without adapter weights.\",\n    )\n    return parser.parse_args()\n\n\ndef load_json(path: Path) -> dict:\n    with path.open(\"r\", encoding=\"utf-8\") as handle:\n        return json.load(handle)\n\n\ndef validate_adapter_dir(adapter_dir: Path, expected_base_model: str, allow_missing_weight_file: bool) -> tuple[dict, list[str]]:\n    errors: list[str] = []\n    for filename in REQUIRED_ADAPTER_FILES:\n        if not (adapter_dir / filename).exists():\n            errors.append(f\"missing required file: {filename}\")\n\n    weight_candidates = [filename for filename in OPTIONAL_WEIGHT_FILES if (adapter_dir / filename).exists()]\n    if not weight_candidates and not allow_missing_weight_file:\n        errors.append(\"missing adapter weights: expected adapter_model.safetensors or adapter_model.bin\")\n\n    config = load_json(adapter_dir / \"adapter_config.json\") if (adapter_dir / \"adapter_config.json\").exists() else {}\n    base_model = str(config.get(\"base_model_name_or_path\", \"\"))\n    if expected_base_model.lower() not in base_model.lower():\n        errors.append(\n            f\"adapter_config.json base_model_name_or_path must contain '{expected_base_model}', got '{base_model or 'MISSING'}'\"\n        )\n    if str(config.get(\"peft_type\", \"\")).upper() != \"LORA\":\n        errors.append(f\"adapter_config.json peft_type must be 'LORA', got '{config.get('peft_type', 'MISSING')}'\")\n    if \"target_modules\" not in config:\n        errors.append(\"adapter_config.json missing target_modules\")\n\n    return config, errors\n\n\ndef copy_submission_files(adapter_dir: Path, submission_dir: Path) -> list[str]:\n    copied: list[str] = []\n    for path in adapter_dir.iterdir():\n        if path.is_file():\n            shutil.copy2(path, submission_dir / path.name)\n            copied.append(path.name)\n    return sorted(copied)\n\n\ndef write_supporting_files(submission_dir: Path, manifest: dict) -> None:\n    (submission_dir / \"README_submission.md\").write_text(\n        \"# Wonderland LoRA submission\\n\\n\"\n        \"This bundle is intended for NVIDIA Nemotron-3-Nano-30B with vLLM-based inference.\\n\\n\"\n        \"## Inference contract\\n\"\n        f\"```text\\n{INFERENCE_CONTRACT}```\\n\",\n        encoding=\"utf-8\",\n    )\n    (submission_dir / \"manifest.json\").write_text(json.dumps(manifest, indent=2) + \"\\n\", encoding=\"utf-8\")\n\n\ndef create_archive(submission_dir: Path) -> Path:\n    archive_path = submission_dir.with_suffix(\".tar.gz\")\n    with tarfile.open(archive_path, \"w:gz\") as archive:\n        archive.add(submission_dir, arcname=submission_dir.name)\n    return archive_path\n\n\ndef main() -> int:\n    args = parse_args()\n    adapter_dir = args.adapter_dir.resolve()\n    output_dir = args.output_dir.resolve()\n    submission_dir = output_dir / args.submission_name\n    output_dir.mkdir(parents=True, exist_ok=True)\n    if submission_dir.exists():\n        shutil.rmtree(submission_dir)\n    submission_dir.mkdir(parents=True, exist_ok=True)\n\n    config, errors = validate_adapter_dir(\n        adapter_dir=adapter_dir,\n        expected_base_model=args.expected_base_model,\n        allow_missing_weight_file=args.allow_missing_weight_file,\n    )\n    if errors:\n        raise SystemExit(\"Submission validation failed:\\n- \" + \"\\n- \".join(errors))\n\n    copied_files = copy_submission_files(adapter_dir, submission_dir)\n    manifest = {\n        \"submission_name\": args.submission_name,\n        \"adapter_dir\": str(adapter_dir),\n        \"copied_files\": copied_files,\n        \"base_model_name_or_path\": config.get(\"base_model_name_or_path\"),\n        \"peft_type\": config.get(\"peft_type\"),\n        \"target_modules\": config.get(\"target_modules\"),\n        \"vllm_assumptions\": {\n            \"base_model\": config.get(\"base_model_name_or_path\"),\n            \"adapter_loaded_via_lora\": True,\n            \"final_answer_emission\": \"single boxed answer with no trailing text\",\n        },\n        \"inference_contract\": INFERENCE_CONTRACT.strip().splitlines(),\n    }\n    write_supporting_files(submission_dir, manifest)\n    archive_path = create_archive(submission_dir)\n\n    print(json.dumps({\n        \"submission_dir\": str(submission_dir),\n        \"archive_path\": str(archive_path),\n        \"copied_files\": copied_files,\n    }, indent=2))\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n"
}''')


def first_existing(paths: list[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None


def is_probable_repo_root(path: Path) -> bool:
    return (path / 'train.csv').exists() and (path / 'scripts').exists()


IS_KAGGLE = KAGGLE_INPUT_ROOT.exists() and first_existing(COMPETITION_INPUT_CANDIDATES) is not None
COMPETITION_INPUT_DIR = first_existing(COMPETITION_INPUT_CANDIDATES)
BASE_MODEL_DIR = first_existing(MODEL_INPUT_CANDIDATES)
REPO_ROOT = (KAGGLE_WORKING_ROOT / 'rag-from-scratch') if IS_KAGGLE else SOURCE_ROOT
NOTEBOOK_ROOT = REPO_ROOT / 'notebooks'
ARTIFACT_ROOT = REPO_ROOT / 'artifacts'
TRAIN_PATH = REPO_ROOT / 'train.csv'
TEST_PATH = REPO_ROOT / 'test.csv'


def ensure_parent(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


def write_embedded_scripts(destination_root: Path) -> None:
    for relative_path, content in EMBEDDED_SCRIPT_SOURCES.items():
        target = destination_root / relative_path
        ensure_parent(target)
        target.write_text(content, encoding='utf-8')


def prepare_kaggle_workspace() -> None:
    if COMPETITION_INPUT_DIR is None:
        raise FileNotFoundError('Kaggle competition input directory was not found.')
    REPO_ROOT.mkdir(parents=True, exist_ok=True)
    for name in ['train.csv', 'test.csv']:
        shutil.copy2(COMPETITION_INPUT_DIR / name, REPO_ROOT / name)
    write_embedded_scripts(REPO_ROOT)


def ensure_local_workspace() -> None:
    if not is_probable_repo_root(REPO_ROOT):
        REPO_ROOT.mkdir(parents=True, exist_ok=True)
        write_embedded_scripts(REPO_ROOT)
    if not TRAIN_PATH.exists() and (SOURCE_ROOT / 'train.csv').exists():
        shutil.copy2(SOURCE_ROOT / 'train.csv', TRAIN_PATH)
    if not TEST_PATH.exists() and (SOURCE_ROOT / 'test.csv').exists():
        shutil.copy2(SOURCE_ROOT / 'test.csv', TEST_PATH)


if IS_KAGGLE:
    prepare_kaggle_workspace()
else:
    ensure_local_workspace()

required_paths = [
    TRAIN_PATH,
    TEST_PATH,
    REPO_ROOT / 'scripts' / 'profile_dataset.py',
    REPO_ROOT / 'scripts' / 'build_router.py',
    REPO_ROOT / 'scripts' / 'eval_baselines.py',
    REPO_ROOT / 'scripts' / 'generate_synthetic.py',
    REPO_ROOT / 'scripts' / 'prepare_sft_data.py',
    REPO_ROOT / 'scripts' / 'package_lora_submission.py',
]

missing = [str(path.relative_to(REPO_ROOT)) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing required workflow paths: {missing}')

print(f'Execution mode: {"Kaggle" if IS_KAGGLE else "Local"}')
print(f'Workspace root: {REPO_ROOT}')
print(f'Python version: {platform.python_version()}')
print(f'Python executable: {sys.executable}')
if COMPETITION_INPUT_DIR is not None:
    print(f'Competition input dir: {COMPETITION_INPUT_DIR}')
if BASE_MODEL_DIR is not None:
    print(f'Base model dir detected: {BASE_MODEL_DIR}')
print('All required notebook inputs are present.')
print('\nWhat to check: Kaggle mode should report a writable /kaggle/working workspace and local mode should keep using the checked-in repository files.')


## 2. Repository discovery

**What this section does:** shows the effective workflow layout used by the notebook and maps each later step to the actual checked-in logic.

This notebook intentionally reuses the existing CLI scripts wherever possible:
- `scripts/profile_dataset.py` for benchmark profiling,
- `scripts/build_router.py` for routing validation,
- `scripts/eval_baselines.py` for the real offline evaluation path,
- `scripts/generate_synthetic.py` for approved synthetic data,
- `scripts/prepare_sft_data.py` for chat-style SFT datasets,
- `scripts/package_lora_submission.py` for packaging validation.

> **Kaggle-specific note:** if those `.py` files are not present as uploaded notebook assets, the notebook writes embedded copies into the writable workspace before executing them. That keeps the workflow self-contained and avoids manual file uploads.


In [ ]:
interesting_dirs = ["docs", "scripts", "src", "configs", "experiments", "outputs", "notebooks"]
for dirname in interesting_dirs:
    path = REPO_ROOT / dirname
    print(f"\n== {dirname}/ exists={path.exists()} ==")
    if path.exists():
        files = sorted(p.relative_to(REPO_ROOT).as_posix() for p in path.rglob("*") if p.is_file())
        for item in files[:50]:
            print(f"  {item}")
        if len(files) > 50:
            print(f"  ... ({len(files) - 50} more files)")

print("\nCore docs reused by this notebook:")
for doc_name in [
    "docs/dataset_forensics.md",
    "docs/task_taxonomy.md",
    "docs/system_architecture.md",
    "docs/eval_plan.md",
    "docs/lora_plan.md",
    "docs/run_from_zero_to_result.md",
]:
    print(f"  - {doc_name}: {'present' if (REPO_ROOT / doc_name).exists() else 'missing'}")

print("\nWhat to check: the repo should show script-centric workflow files under scripts/ and benchmark notes under docs/.")


In [ ]:
def run_command(command: list[str], *, cwd: Path = REPO_ROOT, env: dict[str, str] | None = None, check: bool = True) -> str:
    """Run a workflow command from the prepared workspace and return stdout."""
    print("$", " ".join(command))
    completed = subprocess.run(
        command,
        cwd=cwd,
        env=env,
        text=True,
        capture_output=True,
        check=False,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f"Command failed with code {completed.returncode}: {' '.join(command)}")
    return completed.stdout


def read_json(path: Path):
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


print("Helper utilities loaded.")
print("\nWhat to check: later cells should call run_command(...) against the prepared workspace instead of assuming uploaded Python modules are already available.")


## 3. Quick start path

**What this section does:** runs the shortest path to a first working result using the existing scripts exactly as they are implemented today.

The fastest honest path to a first result is:
1. profile the dataset,
2. validate the router,
3. run the offline baseline evaluation.

Expected outputs:
- terminal summary from `profile_dataset.py`,
- router metrics from `build_router.py`,
- baseline accuracy report from `eval_baselines.py`.


In [ ]:
quick_profile_output = run_command([sys.executable, "scripts/profile_dataset.py"])
quick_router_output = run_command([sys.executable, "scripts/build_router.py", "--skip-demo"])
quick_eval_output = run_command([sys.executable, "scripts/eval_baselines.py", "--baseline", "solver_lite"])

print("Quick-start path finished.")
print("\nWhat to check: profile output should show 9,500 train rows and the router/baseline commands should complete without errors.")


## 4. Data inspection

**What this section does:** inspects the raw CSV inputs directly before any heavier workflow step, matching the repo instructions in `AGENTS.md`.


In [ ]:
for csv_name in ["train.csv", "test.csv"]:
    path = REPO_ROOT / csv_name
    print(f"\n== Preview: {csv_name} ==")
    with path.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        for row in islice(reader, 2):
            preview = {
                key: (value[:160] + "..." if isinstance(value, str) and len(value) > 160 else value)
                for key, value in row.items()
            }
            pprint(preview)

print("\nWhat to check: confirm the benchmark really is prompt/answer data and that test.csv has no answer column.")


## 5. End-to-end pipeline

**What this section does:** runs the full practical workflow supported by the checked-in repository from dataset profiling through optional packaging.

The notebook stays aligned to the actual codebase:
- it uses subprocess calls to the checked-in scripts,
- it avoids reimplementing solver logic inside notebook cells,
- and it clearly labels optional sections where the repo depends on external training or adapter artifacts.


### 5.1 Dataset profiling

**What this section does:** materializes a machine-readable dataset profile that can be saved and compared across future runs.


In [ ]:
ARTIFACT_ROOT.mkdir(exist_ok=True)
profile_json_path = ARTIFACT_ROOT / "profile_dataset_notebook.json"
profile_stdout = run_command([sys.executable, "scripts/profile_dataset.py", "--json"])
profile_json_path.write_text(profile_stdout, encoding="utf-8")
profile = json.loads(profile_stdout)

print(f"Saved machine-readable profile to: {profile_json_path.relative_to(REPO_ROOT)}")
print("Train rows:", profile["overall"]["train_rows"])
print("Visible test rows:", profile["overall"]["test_rows"])
print("Exact visible test/train overlap:", profile["overall"]["exact_test_prompt_overlap_with_train"])
print("Families:", sorted(profile["families"].keys()))
print("\nWhat to check: the saved JSON should exist under artifacts/ and confirm that visible test rows overlap with train.")


### 5.2 Router validation

**What this section does:** runs the real family-router evaluation path and preserves the text output for later comparison.


In [ ]:
router_report_path = ARTIFACT_ROOT / "router_eval_notebook.txt"
router_stdout = run_command([sys.executable, "scripts/build_router.py", "--skip-demo"])
router_report_path.write_text(router_stdout, encoding="utf-8")

print(f"Saved router report to: {router_report_path.relative_to(REPO_ROOT)}")
print("\nWhat to check: top-level routing accuracy should be 1.0000 on the visible train split.")


### 5.3 Offline evaluation

**What this section does:** runs the repository's real evaluation script as a small fixed-setting baseline matrix, saves separate reports, and extracts the most important metrics for quick review.

> **Fair-comparison note:** keep the split strategy and fold count fixed when comparing baselines. In this notebook, every baseline below uses the same `scripts/eval_baselines.py` split logic and the same `--num-folds` value.


In [ ]:
NOTEBOOK_BASELINES = ["last_demo", "solver_lite"]
NOTEBOOK_EVAL_NUM_FOLDS = 5
NOTEBOOK_SPLITS = ["stratified_random_hash", "structure_aware_hash"]
NOTEBOOK_FAMILIES = [
    "bit_transform",
    "text_cipher",
    "roman_numeral",
    "unit_conversion",
    "gravity",
    "equation_transform",
]


def parse_baseline_report(report_text: str) -> dict[str, object]:
    parsed: dict[str, object] = {"splits": {}, "random_vs_structure_accuracy_gap": None}
    current_split = None
    in_family_block = False

    for raw_line in report_text.splitlines():
        stripped = raw_line.strip()
        if not stripped:
            continue
        if stripped.startswith("Baseline:"):
            parsed["baseline"] = stripped.split(":", 1)[1].strip()
            continue
        if stripped.startswith("Rows evaluated:"):
            parsed["rows_evaluated"] = int(stripped.split(":", 1)[1].strip())
            continue
        if stripped.endswith(":") and stripped[:-1] in NOTEBOOK_SPLITS:
            current_split = stripped[:-1]
            parsed["splits"][current_split] = {"per_family_accuracy": {}}
            in_family_block = False
            continue
        if stripped.startswith("random_vs_structure_accuracy_gap="):
            parsed["random_vs_structure_accuracy_gap"] = float(stripped.split("=", 1)[1])
            continue
        if current_split is None:
            continue
        if stripped.startswith("overall_accuracy="):
            parsed["splits"][current_split]["overall_accuracy"] = float(stripped.split("=", 1)[1].split()[0])
            continue
        if stripped.startswith("answer_format_accuracy="):
            parsed["splits"][current_split]["answer_format_accuracy"] = float(stripped.split("=", 1)[1].split()[0])
            continue
        if stripped == "per_family_accuracy:":
            in_family_block = True
            continue
        if in_family_block and stripped.startswith("-"):
            family, score = stripped[1:].split()
            parsed["splits"][current_split]["per_family_accuracy"][family] = float(score)
            continue
        if in_family_block and not stripped.startswith("-"):
            in_family_block = False

    return parsed


def render_baseline_comparison_table(rows: list[dict[str, object]]) -> str:
    headers = ["baseline", "split", "overall_acc", "format_acc", "per_family_acc"]
    table_rows = []
    for row in rows:
        family_summary = ", ".join(
            f"{family}={row['per_family_accuracy'][family]:.4f}" for family in NOTEBOOK_FAMILIES
        )
        table_rows.append([
            row["baseline"],
            row["split"],
            f"{row['overall_accuracy']:.4f}",
            f"{row['answer_format_accuracy']:.4f}",
            family_summary,
        ])

    widths = [len(header) for header in headers]
    for table_row in table_rows:
        for idx, value in enumerate(table_row):
            widths[idx] = max(widths[idx], len(str(value)))

    def fmt(values: list[str]) -> str:
        return " | ".join(str(value).ljust(widths[idx]) for idx, value in enumerate(values))

    separator = "-+-".join("-" * width for width in widths)
    lines = [fmt(headers), separator]
    lines.extend(fmt(row) for row in table_rows)
    return "\n".join(lines)


baseline_reports: dict[str, dict[str, object]] = {}
comparison_rows: list[dict[str, object]] = []

for baseline_name in NOTEBOOK_BASELINES:
    eval_report_path = ARTIFACT_ROOT / f"baseline_eval_{baseline_name}.txt"
    eval_stdout = run_command(
        [
            sys.executable,
            "scripts/eval_baselines.py",
            "--baseline",
            baseline_name,
            "--num-folds",
            str(NOTEBOOK_EVAL_NUM_FOLDS),
        ]
    )
    eval_report_path.write_text(eval_stdout, encoding="utf-8")
    parsed_report = parse_baseline_report(eval_stdout)
    parsed_report["report_path"] = eval_report_path.relative_to(REPO_ROOT).as_posix()
    baseline_reports[baseline_name] = parsed_report

    for split_name in NOTEBOOK_SPLITS:
        split_metrics = parsed_report["splits"][split_name]
        comparison_rows.append(
            {
                "baseline": baseline_name,
                "split": split_name,
                "overall_accuracy": split_metrics["overall_accuracy"],
                "answer_format_accuracy": split_metrics["answer_format_accuracy"],
                "per_family_accuracy": {
                    family: split_metrics["per_family_accuracy"].get(family)
                    for family in NOTEBOOK_FAMILIES
                },
            }
        )

baseline_comparison = {
    "num_folds": NOTEBOOK_EVAL_NUM_FOLDS,
    "split_strategies": NOTEBOOK_SPLITS,
    "baseline_order": NOTEBOOK_BASELINES,
    "reports": baseline_reports,
    "comparison_rows": comparison_rows,
}

comparison_json_path = ARTIFACT_ROOT / "baseline_eval_comparison_notebook.json"
comparison_json_path.write_text(json.dumps(baseline_comparison, indent=2, sort_keys=True), encoding="utf-8")
comparison_table = render_baseline_comparison_table(comparison_rows)

print(json.dumps(baseline_comparison, indent=2, sort_keys=True))
print("\nCompact baseline comparison table:")
print(comparison_table)
for baseline_name in NOTEBOOK_BASELINES:
    report_path = baseline_reports[baseline_name]["report_path"]
    print(f"Saved {baseline_name} report to: {report_path}")
print(f"Saved comparison object to: {comparison_json_path.relative_to(REPO_ROOT)}")
print("\nWhat to check: solver_lite should still land around 0.4336 overall / 1.0000 format accuracy, while last_demo provides the lower reference point under the same split settings.")


### 5.4 Synthetic data generation

**What this section does:** materializes the notebook's full documented synthetic ladder, including an explicit no-synthetic marker plus the approved `mvp` and `full` presets from the checked-in generator.

This stays within the repo's approved synthetic scope:
- `roman_numeral`
- `unit_conversion`
- `gravity`

The goal here is provenance-first experiment setup rather than a single MVP-only path.


In [ ]:
synthetic_root = ARTIFACT_ROOT / "synthetic"
synthetic_root.mkdir(parents=True, exist_ok=True)

synthetic_specs = [
    {
        "synthetic_source": "none",
        "preset": None,
        "path": synthetic_root / "synthetic_train_notebook_none.marker.json",
    },
    {
        "synthetic_source": "mvp",
        "preset": "mvp",
        "path": synthetic_root / "synthetic_train_notebook_mvp.csv",
    },
    {
        "synthetic_source": "full",
        "preset": "full",
        "path": synthetic_root / "synthetic_train_notebook_full.csv",
    },
]

synthetic_artifacts = {}
for spec in synthetic_specs:
    synthetic_source = spec["synthetic_source"]
    artifact_path = spec["path"]
    if spec["preset"] is None:
        marker_payload = {
            "synthetic_source": synthetic_source,
            "preset": None,
            "row_count": 0,
            "family_counts": {},
            "note": "Explicit no-synthetic baseline marker for notebook experiment provenance.",
        }
        artifact_path.write_text(json.dumps(marker_payload, indent=2) + "\n", encoding="utf-8")
        synthetic_artifacts[synthetic_source] = {
            **marker_payload,
            "path": artifact_path,
        }
        continue

    run_command([
        sys.executable,
        "scripts/generate_synthetic.py",
        "--preset", spec["preset"],
        "--output", str(artifact_path),
        "--seed", "7",
    ])

    with artifact_path.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    family_counts = {}
    for row in rows:
        family_counts[row["family"]] = family_counts.get(row["family"], 0) + 1
    synthetic_artifacts[synthetic_source] = {
        "synthetic_source": synthetic_source,
        "preset": spec["preset"],
        "row_count": len(rows),
        "family_counts": family_counts,
        "path": artifact_path,
    }

synthetic_summary_path = synthetic_root / "notebook_synthetic_ladder_summary.json"
synthetic_summary_payload = [
    {
        "synthetic_source": source,
        "preset": info["preset"],
        "row_count": info["row_count"],
        "family_counts": info["family_counts"],
        "path": str(info["path"].relative_to(REPO_ROOT)),
    }
    for source, info in synthetic_artifacts.items()
]
synthetic_summary_path.write_text(json.dumps(synthetic_summary_payload, indent=2) + "\n", encoding="utf-8")

print("Synthetic ladder artifacts:")
for source, info in synthetic_artifacts.items():
    print(
        f"  - {source:4s} :: rows={info['row_count']:5d} :: "
        f"path={info['path'].relative_to(REPO_ROOT)}"
    )
    if info["family_counts"]:
        print(f"      family_counts={info['family_counts']}")

print(f"\nSaved synthetic ladder summary: {synthetic_summary_path.relative_to(REPO_ROOT)}")
print("\nWhat to check: the notebook should materialize an explicit no-synthetic marker plus distinct MVP and full synthetic CSVs under artifacts/synthetic/.")


### 5.5 SFT data preparation

**What this section does:** materializes the notebook's full documented SFT experiment ladder across every supported recipe from `scripts/prepare_sft_data.py`.

The notebook keeps provenance explicit by:
- running all supported recipes,
- separating `none` / `mvp` / `full` synthetic sources into distinct output directories,
- only adding `boxed` vs `plain` variants where the docs explicitly call for a boxed-target comparison, and
- recording train/dev counts in a notebook summary table for every generated dataset.


In [ ]:
sft_output_dir = ARTIFACT_ROOT / "sft_notebook"
sft_output_dir.mkdir(parents=True, exist_ok=True)

sft_experiment_specs = [
    {
        "synthetic_source": "none",
        "recipe": "original_sft",
        "box_style": "boxed",
        "max_synthetic_cap": 0,
    },
    {
        "synthetic_source": "mvp",
        "recipe": "mixed_sft",
        "box_style": "boxed",
        "max_synthetic_cap": 0,
    },
    {
        "synthetic_source": "mvp",
        "recipe": "mixed_sft",
        "box_style": "plain",
        "max_synthetic_cap": 0,
    },
    {
        "synthetic_source": "full",
        "recipe": "mixed_sft",
        "box_style": "boxed",
        "max_synthetic_cap": 0,
    },
    {
        "synthetic_source": "mvp",
        "recipe": "family_conditioned",
        "box_style": "boxed",
        "max_synthetic_cap": 0,
    },
    {
        "synthetic_source": "full",
        "recipe": "family_conditioned",
        "box_style": "boxed",
        "max_synthetic_cap": 0,
    },
    {
        "synthetic_source": "none",
        "recipe": "format_stabilization",
        "box_style": "boxed",
        "max_synthetic_cap": 0,
    },
    {
        "synthetic_source": "mvp",
        "recipe": "format_stabilization",
        "box_style": "boxed",
        "max_synthetic_cap": 0,
    },
    {
        "synthetic_source": "full",
        "recipe": "format_stabilization",
        "box_style": "boxed",
        "max_synthetic_cap": 0,
    },
]

sft_experiment_records = []
for spec in sft_experiment_specs:
    output_subdir = sft_output_dir / (
        f"{spec['recipe']}_{spec['synthetic_source']}_{spec['box_style']}_cap{spec['max_synthetic_cap']}"
    )
    output_subdir.mkdir(parents=True, exist_ok=True)

    command = [
        sys.executable,
        "scripts/prepare_sft_data.py",
        "--output-dir", str(output_subdir),
        "--recipe", spec["recipe"],
        "--box-style", spec["box_style"],
        "--max-synthetic-per-family", str(spec["max_synthetic_cap"]),
    ]
    if spec["synthetic_source"] != "none":
        command.extend([
            "--synthetic-path",
            str(synthetic_artifacts[spec["synthetic_source"]]["path"]),
        ])

    run_command(command)

    summary_path = output_subdir / f"{spec['recipe']}_{spec['box_style']}.summary.json"
    summary = read_json(summary_path)
    sft_experiment_records.append({
        "synthetic_source": spec["synthetic_source"],
        "recipe": spec["recipe"],
        "box_style": spec["box_style"],
        "max_synthetic_cap": spec["max_synthetic_cap"],
        "output_path": str(output_subdir.relative_to(REPO_ROOT)),
        "train_rows": summary["train_rows"],
        "dev_rows": summary["dev_rows"],
        "summary_path": str(summary_path.relative_to(REPO_ROOT)),
        "train_path": str(Path(summary["train_path"]).relative_to(REPO_ROOT)),
        "dev_path": str(Path(summary["dev_path"]).relative_to(REPO_ROOT)),
    })

sft_summary_path = sft_output_dir / "notebook_sft_experiment_ladder_summary.json"
sft_summary_path.write_text(json.dumps(sft_experiment_records, indent=2) + "\n", encoding="utf-8")

summary_headers = [
    "synthetic_source",
    "recipe",
    "box_style",
    "max_synthetic_cap",
    "output_path",
    "train_rows",
    "dev_rows",
]
summary_widths = {
    header: max(len(header), max(len(str(row[header])) for row in sft_experiment_records))
    for header in summary_headers
}

def format_row(row):
    return " | ".join(str(row[header]).ljust(summary_widths[header]) for header in summary_headers)

print("Notebook SFT experiment ladder summary:")
print(format_row({header: header for header in summary_headers}))
print("-+-".join("-" * summary_widths[header] for header in summary_headers))
for row in sft_experiment_records:
    print(format_row(row))

sample_record_path = Path(sft_experiment_records[0]["train_path"])
with sample_record_path.open("r", encoding="utf-8") as handle:
    first_record = json.loads(handle.readline())
print("\nSample SFT record metadata from the first generated dataset:")
pprint(first_record["metadata"])
print(f"\nSaved SFT ladder summary: {sft_summary_path.relative_to(REPO_ROOT)}")
print("\nWhat to check: every supported SFT recipe should be materialized, the mixed-SFT MVP boxed/plain comparison should both exist, and each run should live in a provenance-explicit output directory.")


### 5.6 Inference / result generation status

**What this section does:** makes the current repository boundary explicit so a new engineer does not assume missing functionality exists.

> **Current repo gap:** there is no checked-in training script, no vLLM inference launcher, and no script that reads `test.csv` and emits final hidden-test predictions. In this repo version, the practical top-to-bottom path ends at offline evaluation, training-data preparation, and optional submission packaging.

The "result generation" step that is currently implemented is therefore:
- offline baseline result generation via `scripts/eval_baselines.py`, and
- artifact generation for later LoRA work via `scripts/generate_synthetic.py` and `scripts/prepare_sft_data.py`.


### 5.7 Optional submission packaging

**What this section does:** demonstrates the real packaging path. If you already have an adapter directory, point the notebook to it. Otherwise, the notebook creates a minimal **config-only dry-run adapter** so the packaging workflow can still be exercised safely.


In [ ]:
user_adapter_dir = os.environ.get("NOTEBOOK_ADAPTER_DIR", "").strip()
if user_adapter_dir:
    adapter_dir = Path(user_adapter_dir).expanduser().resolve()
    allow_missing = False
    print(f"Using existing adapter directory: {adapter_dir}")
else:
    adapter_dir = ARTIFACT_ROOT / "demo_adapter"
    adapter_dir.mkdir(parents=True, exist_ok=True)
    demo_config = {
        "base_model_name_or_path": "nvidia/Nemotron-3-Nano-30B-Instruct",
        "peft_type": "LORA",
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
        "r": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.0,
    }
    (adapter_dir / "adapter_config.json").write_text(json.dumps(demo_config, indent=2) + "\n", encoding="utf-8")
    allow_missing = True
    print(f"Created config-only demo adapter at: {adapter_dir.relative_to(REPO_ROOT)}")

submission_output_dir = ARTIFACT_ROOT / "submission_notebook"
command = [
    sys.executable,
    "scripts/package_lora_submission.py",
    "--adapter-dir", str(adapter_dir),
    "--output-dir", str(submission_output_dir),
    "--submission-name", "notebook_demo_submission",
]
if allow_missing:
    command.append("--allow-missing-weight-file")

package_stdout = run_command(command)
package_info = json.loads(package_stdout)
pprint(package_info)

manifest_path = Path(package_info["submission_dir"]) / "manifest.json"
manifest = read_json(manifest_path)
print("\nManifest keys:", sorted(manifest.keys()))
print("\nWhat to check: packaging should produce a submission directory, a tar.gz archive, and a manifest.json file.")


## 6. Evaluation guidance

**What this section does:** explains how to interpret the repo's real evaluation output and how to compare experiments fairly.

### What the reported metrics mean
- **overall accuracy**: exact-match accuracy across all evaluated rows.
- **answer-format accuracy**: whether the prediction matches the expected output schema, even when the answer itself is wrong.
- **per-family accuracy**: family-by-family exact-match performance.
- **random vs structure-aware comparison**: whether the method generalizes across latent structural buckets instead of only surface-near rows.

### Fair experiment comparison rules in this repo
1. Compare runs on the **same baseline script and split settings**.
2. Do **not** use the visible `test.csv` as a model-selection target.
3. Keep validation **real-only** when synthetic rows are introduced for training.
4. Compare both:
   - exact match, and
   - answer-format accuracy.
5. Review per-family movement, especially for the unsolved families:
   - `bit_transform`
   - `text_cipher`
   - `equation_transform`

### Warning about misleading evaluation
- The visible `test.csv` is fully overlapped with train and should only be treated as a smoke test.
- Strong gains on easy deterministic families can hide the fact that hard families remain unsolved.
- Format-only gains can look deceptively good if they do not improve exact match.
- Structure-aware comparisons are more informative than random splits when hidden-test drift is possible.


In [ ]:
print("Parsed evaluation summary by baseline:")
for baseline_name in baseline_comparison["baseline_order"]:
    report = baseline_comparison["reports"][baseline_name]
    print(f"\n== {baseline_name} ==")
    for split_name in baseline_comparison["split_strategies"]:
        metrics = report["splits"][split_name]
        print(f"  [{split_name}]")
        print(f"    overall_accuracy       = {metrics['overall_accuracy']:.4f}")
        print(f"    answer_format_accuracy = {metrics['answer_format_accuracy']:.4f}")
        print("    per_family_accuracy    =")
        for family in NOTEBOOK_FAMILIES:
            score = metrics["per_family_accuracy"][family]
            print(f"      - {family:18s} {score:.4f}")

hard_families = ["bit_transform", "text_cipher", "equation_transform"]
print("\nHard-family snapshot on stratified_random_hash:")
for baseline_name in baseline_comparison["baseline_order"]:
    report = baseline_comparison["reports"][baseline_name]
    print(f"  {baseline_name}:")
    for family in hard_families:
        score = report["splits"]["stratified_random_hash"]["per_family_accuracy"][family]
        print(f"    {family:18s} {score:.4f}")

print("\nWhat to check: easy-family accuracy should still dominate today, while the hard-family deltas between last_demo and solver_lite show where solver logic is already helping.")


## 7. Results inspection and validation

**What this section does:** shows where outputs were written, how to inspect them, and how to tell whether the run actually succeeded.


In [ ]:
expected_artifacts = [
    profile_json_path,
    router_report_path,
    *(ARTIFACT_ROOT / f"baseline_eval_{baseline_name}.txt" for baseline_name in NOTEBOOK_BASELINES),
    comparison_json_path,
    *(info["path"] for info in synthetic_artifacts.values()),
    synthetic_summary_path,
    sft_summary_path,
    *(Path(record["summary_path"]) for record in sft_experiment_records),
    *(Path(record["train_path"]) for record in sft_experiment_records),
    *(Path(record["dev_path"]) for record in sft_experiment_records),
    submission_output_dir / "notebook_demo_submission",
    submission_output_dir / "notebook_demo_submission.tar.gz",
]

print("Generated artifacts:")
for artifact in expected_artifacts:
    print(f"  - {artifact.relative_to(REPO_ROOT)} :: exists={artifact.exists()}")

print("
SFT experiment ladder snapshot:")
for record in sft_experiment_records:
    print(
        f"  - {record['recipe']:20s} :: source={record['synthetic_source']:4s} :: "
        f"box={record['box_style']:5s} :: train={record['train_rows']:5d} :: dev={record['dev_rows']:4d}"
    )

print("
Common failure patterns to watch for:")
print("  1. Running the notebook from the wrong working directory.")
print("  2. Assuming test.csv is a true validation set.")
print("  3. Expecting a checked-in training or inference runner that does not exist yet.")
print("  4. Forgetting that packaging requires a real adapter directory unless dry-run mode is used.")
print("  5. Comparing experiments without keeping the evaluation split strategy and fold count fixed.")
print("  6. Mixing synthetic-source or box-style variants without preserving directory provenance.")

print("
What to check: every expected artifact above should exist after a successful top-to-bottom run, including the full synthetic and SFT experiment ladders.")


## 8. Reproducibility and next steps

**What this section does:** records what should be saved from a run and suggests the highest-value next experiment path based on the actual repository state.

### Save these artifacts
- `artifacts/profile_dataset_notebook.json`
- `artifacts/router_eval_notebook.txt`
- `artifacts/baseline_eval_last_demo.txt`
- `artifacts/baseline_eval_solver_lite.txt`
- `artifacts/baseline_eval_comparison_notebook.json`
- notebook-rendered compact comparison table from `baseline_comparison["comparison_rows"]`
- `artifacts/synthetic/synthetic_train_notebook_none.marker.json`
- `artifacts/synthetic/synthetic_train_notebook_mvp.csv`
- `artifacts/synthetic/synthetic_train_notebook_full.csv`
- `artifacts/synthetic/notebook_synthetic_ladder_summary.json`
- `artifacts/sft_notebook/notebook_sft_experiment_ladder_summary.json`
- all per-run SFT directories under `artifacts/sft_notebook/`, including:
  - `original_sft_none_boxed_cap0/`
  - `mixed_sft_mvp_boxed_cap0/`
  - `mixed_sft_mvp_plain_cap0/`
  - `mixed_sft_full_boxed_cap0/`
  - `family_conditioned_mvp_boxed_cap0/`
  - `family_conditioned_full_boxed_cap0/`
  - `format_stabilization_none_boxed_cap0/`
  - `format_stabilization_mvp_boxed_cap0/`
  - `format_stabilization_full_boxed_cap0/`
- `artifacts/submission_notebook/notebook_demo_submission*`

### What to pin
- Python version
- notebook execution order
- synthetic generation seed
- the synthetic ladder (`none`, `mvp`, `full`)
- the SFT recipe/box-style matrix captured in `notebook_sft_experiment_ladder_summary.json`
- evaluation baseline matrix (`last_demo` and `solver_lite` unless intentionally expanded)
- evaluation fold count (`NOTEBOOK_EVAL_NUM_FOLDS`)
- the fixed split strategies reported by `scripts/eval_baselines.py`
- any future training config, LoRA rank, and adapter base model once training exists

### How to rerun fairly
- rerun the notebook from the same effective workspace (local repo root or Kaggle workspace),
- keep the same seeds and CLI flags,
- keep the same synthetic-source and box-style ladder,
- keep the split strategy and fold count fixed across baseline comparisons,
- avoid changing evaluation settings between comparison runs,
- compare the saved baseline reports plus the structured synthetic/SFT ladder summaries side by side.

### Suggested next experiment path
Based on the current repo implementation, the next highest-value steps are:
1. build stronger solver coverage for `bit_transform`,
2. reverse-engineer the `text_cipher` family,
3. deepen `equation_transform` taxonomy and solver search,
4. then train the smallest boxed-answer-stabilization LoRA that earns a real-only validation gain over the prompt-only baseline,
5. and only expand synthetic-conditioned LoRA work if the notebook's ladder artifacts show robust gains instead of format-only movement.


## Definition of Success

**What this section does:** summarizes the expected outputs, the key checks that indicate success, and the most common reasons a top-to-bottom run may fail.

A notebook run is successful if it produces the following practical outputs:
- a saved dataset profile JSON,
- a saved router evaluation report,
- saved offline baseline matrix reports plus the comparison JSON artifact,
- the full synthetic ladder artifacts for `none`, `mvp`, and `full`,
- the full documented SFT experiment ladder across `original_sft`, `mixed_sft`, `family_conditioned`, and `format_stabilization`,
- a notebook summary table plus JSON summary for every generated SFT dataset,
- and a submission package directory plus archive when packaging is exercised.

### Expected checks
- dataset profiling confirms the visible `test.csv` is only a smoke-test sample,
- router validation completes successfully,
- baseline matrix evaluation reports overall accuracy, answer-format accuracy, and per-family accuracy,
- synthetic generation writes the explicit no-synthetic marker plus distinct MVP and full synthetic CSVs,
- SFT preparation writes every planned provenance-explicit dataset directory, including the MVP mixed-SFT boxed/plain comparison required by the docs,
- the SFT summary table records synthetic source, recipe, box style, max synthetic cap, output path, and train/dev row counts for every generated dataset,
- and packaging writes `manifest.json` and an archive.

### Common reasons the run may fail
- the notebook cannot prepare its effective workspace or locate the competition input files,
- required CSV or script files are missing,
- a user expects missing training/inference functionality that is not checked in,
- a real adapter directory is not available and dry-run packaging is not used,
- experiment comparisons are made against the leaked visible `test.csv` instead of the real offline evaluation path,
- or synthetic-source / box-style variants are overwritten because their output directories are not kept distinct.
